# C1 · PSF cromática — notebook de análisis (`debug`)

**Objeto:** ROXs42Bb  |  **Run:** `ROXs42Bb_realigned`  |  **Spec:** [`docs/spec_C1_codex_chromatic_psf.md`](../../../docs/spec_C1_codex_chromatic_psf.md)

Rehace C1 **dentro del notebook**. C1 no entrega un espectro: entrega `psf_model.json`, el modelo de PSF que usan **C2, C3, C4, C5, C6, D2, E1b, E2, E4 y E5**. Un sesgo aquí no se queda aquí.

La etapa tiene dos mitades, y este notebook las trata distinto:

1. **El ajuste por bin** (§6) es caro —una ventana de ~100 Å por bin, cada una un ajuste completo— así que va submuestreado con una perilla. Los bins que se reajustan salen idénticos a los de la cadena: cada bin se ajusta solo.
2. **El ensamblado y la evaluación del modelo** (§8, §12) son baratos y se hacen **enteros y exactos**. El constructor del documento es una función pura de las filas por bin, y esas filas están en los CSV de la etapa: el notebook reconstruye `psf_model.json` desde el CSV sin tocar el cubo, y debe salir idéntico.

> **Las dos formas viajan.** C1 ajusta Moffat y Psfao siempre y se queda con la del menor residuo de anillo, y esa elección **depende del objeto**: ROXs 12 b sale `psfao` y ROXs 42B b sale `moffat`. El notebook detecta cuál ganó y audita la que de verdad se entregó, diciendo qué secciones no aplican.

> **La sección que importa es la §8.** El QC de C1 dice que los siete parámetros van suavizados con `polynomial_deg2`, y eso induce a pensar que el modelo por canal es una parábola. **No lo es.** Si el documento trae `param_table` —y lo trae— `_evaluate_psfao` interpola la tabla por bin y `smoothed_poly` no interviene. La §8 lo enseña con número, y enseña de dónde salen los saltos escalonados: la λ redondeada a 50 Å, los huecos de la tabla, y los recortes.


In [ ]:
import csv, json, sys
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

# Resolución de las figuras EN PANTALLA. `savefig` guarda a 300 dpi, pero
# lo que se ve dentro del notebook lo fija el backend inline, que va a 100
# dpi por defecto y sale borroso. `retina` dobla los píxeles sin cambiar el
# tamaño aparente; fuera de IPython no hace nada y queda el rcParam.
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['savefig.dpi'] = 200
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
_here = Path.cwd()
ROOT = next(p for p in (_here, *_here.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)
print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)


## 1 · Perillas

Todo sale del config **resuelto** de la etapa: la etapa rellena defaults que el run no escribe, y copiarlos como literales es exactamente lo que hizo que el primer notebook de C3 no reprodujera la cadena.

Ojo con `psf_fit_radius_px`: el módulo de psfao tiene `80.0` escrito como default propio, pero nunca lo usa, porque recibe el config **ya rellenado** por `stage_e01_config_from_run`, que pone `28.0`. El número que manda es el de aquí.


In [ ]:
from musepipe.stages.stage_e01_psf import stage_e01_config_from_run, stage_e01_paths

# `project_root=ROOT`: musepipe resuelve rutas contra el cwd, que en un
# notebook es su propia carpeta, no la raíz del repo.
E01 = stage_e01_config_from_run(RUN_ID, project_root=ROOT)
PATHS_E01 = stage_e01_paths(RUN_ID, project_root=ROOT)

BIN_A            = float(E01.get('psf_bin_A', 100.0))
MIN_CANALES_BIN  = int(E01.get('psf_min_channels_per_bin', 3))
FIT_RADIUS_PX    = float(E01.get('psf_fit_radius_px', 80.0))
NORM_RADIUS_PX   = float(E01.get('psf_norm_radius_px', 25.0))
RING_WIDTH_PX    = float(E01.get('psf_companion_ring_width_px', 3.0))
FORMA            = str(E01.get('e01_psf_form', 'auto')).lower()
BAD_WINDOWS_MOFFAT = E01.get('stage_e01_bad_windows_A', [])
MODO_INSTRUMENTO = str(E01.get('instrument_mode', 'NFM'))

# ---- a partir de aquí, cambia lo que quieras probar ----

# Submuestreo del ajuste por bin (§6): 1 de cada N. Cada bin se ajusta de
# forma independiente, así que los que se reajustan salen idénticos a los de
# la cadena. Pon 1 para recorrer los 43 (minutos).
PASO_BINS = 8
# Y además unos cuantos de los que la cadena marcó como fallidos, para ver
# el modo de fallo reproducido y no solo contado.
INCLUIR_FALLIDOS = 2
# Radios de la curva de crecimiento de la §9, en píxeles.
RADIOS_PX = (10.0, 15.0, 20.0, 25.0, 32.0, 40.0)

# ---- perillas de la §10 (los parámetros cubo a cubo) ----
# Medido en esta máquina: un ajuste Psfao cuesta ~4.5 s y uno Moffat ~0.3 s.
# Con 29 cubos por exposición, 1 de cada 4 bins son ~320 ajustes (~24 min la
# PRIMERA vez); PASO_BINS_MAPA=1 da la rejilla completa de 43 columnas por
# ~96 min. Se paga una sola vez: el resultado va a una caché en disco.
PASO_BINS_MAPA = 4
# `None` = todos los cubos que declare el config. Un número los recorta (los
# primeros por fecha), que es lo cómodo para probar la sección.
MAX_CUBOS = None
# Ancho de bin del barrido. Por defecto el de la cadena, para que las columnas
# del mapa signifiquen lo mismo que los bins del CSV. Súbelo si los ajustes por
# exposición salen estancados: una exposición suelta tiene ~1/N de la
# profundidad del combinado, y un bin más ancho está mejor condicionado.
BIN_A_MAPA = BIN_A
# Fuerza el recálculo aunque la caché valga.
RECALCULAR_MAPA = False
# Los dos cubos de las curvas de la §10.b. `None` = el primero de cada noche.
CUBOS_LINEAS = None
# Procesos para el barrido. Va en 1 porque el reparto NO se ha demostrado
# que ayude: con 8 procesos el ajuste de Psfao dio ×1.6, incluso con la
# carga equilibrada y BLAS a un hilo. Pero la máquina tenía otro trabajo
# encima cuando se midió (load ~5 de 8 núcleos), así que ese ×1.6 está
# contaminado y no es un veredicto: si vas a subirlo, MÍDELO en una caja
# ociosa antes de creerte el número.
N_TRABAJOS = 1

print(f'bins de {BIN_A:.0f} Å (>= {MIN_CANALES_BIN} canales)'
      f' | ajuste r={FIT_RADIUS_PX:.0f} px · normalización r={NORM_RADIUS_PX:.0f} px')
print(f'forma: {FORMA} | 1 de cada {PASO_BINS} bins + {INCLUIR_FALLIDOS} fallidos')


## 2 · Entradas y productos de la cadena

C1 ajusta sobre el **stack de B2** (`stage02_xcorr_cube_stack.fits`) y toma las posiciones de B3. Deja tres productos: los dos CSV por bin (uno por forma) y el `psf_model.json` que consume todo lo demás.

La celda comprueba además que el modelo en disco sea **más nuevo que el cubo sobre el que dice haberse ajustado**. Si el cubo se cambió después, lo que hay en `stages/` describe una PSF que ya no es la de estos datos, y las secciones §8–§12 estarían auditando un modelo huérfano. Avisa; no falla.


In [ ]:
import datetime as _dt

QC_C1 = json.loads((SD / 'stage_e01_qc.json').read_text(encoding='utf-8'))
PSF_MODEL = json.loads((SD / 'psf_model.json').read_text(encoding='utf-8'))
POS = json.loads(Path(E01.get('stage_e01_positions_qc',
                             SD / 'stage01c_qc.json')).read_text(encoding='utf-8'))
# Lo mismo que hace `_positions_from_qc`: desempaquetar el QC de B3. Es
# lectura, no cálculo, así que va como código plano igual que en C2-C4.
STAR_YX = tuple(map(float, POS['primary']['pos_yx']))
COMP_YX = tuple(map(float, POS['companion']['pos_yx']))
FIELD_YX = (None if POS.get('field_source') is None
            else tuple(map(float, POS['field_source']['pos_yx'])))

CUBE_PATH = Path(E01.get('stage_e01_input_cube_fits',
                         SD / 'stage02_xcorr_cube_stack.fits'))
with fits.open(CUBE_PATH, memmap=True) as h:
    WAVE_ALL = np.asarray(h['WAVELENGTH'].data, dtype=float)
    FORMA_CUBO = h['CUBES'].shape if 'CUBES' in h else h[1].shape

# Las mismas radios/máscaras que deriva `prepare_psfao_inputs` (es preparación,
# no cálculo: por eso va aquí y no en la copia).
FWHM_PRELIM = (float(POS.get('companion', {}).get('err_px', 0) or 0)
               + float(E01.get('psf_prelim_fwhm_px', 4.0)))
MASK_RADIUS_PX = float(E01.get('psf_companion_mask_radius_px',
                               max(10.0, 3.0 * FWHM_PRELIM)))
from maoppy.instrument import muse_nfm, muse_wfm
SYSTEM = muse_wfm if MODO_INSTRUMENTO.upper().startswith('W') else muse_nfm
SYSTEM_NAME = 'muse_wfm' if SYSTEM is muse_wfm else 'muse_nfm'

def _filas_csv(path):
    """Las filas por bin, con los números como números.

    `_write_psfao_csv` escribe con `str(float)`, que es `repr`: el ida y
    vuelta por texto es EXACTO, y por eso la §12 puede reconstruir el modelo
    desde el CSV y exigir igualdad bit a bit.
    """
    filas = []
    with open(path, newline='', encoding='utf-8') as f:
        lector = csv.DictReader(f, restkey='_sobrante')
        for cruda in lector:
            if cruda.get('_sobrante'):
                raise ValueError(f'{Path(path).name}: fila con comas de más '
                                 '(el escritor no entrecomilla); revísala a mano')
            fila = {}
            for k, v in cruda.items():
                if v is None or v == '':
                    continue
                if v in ('True', 'False'):
                    fila[k] = (v == 'True')
                    continue
                try:
                    fila[k] = float(v)
                except ValueError:
                    fila[k] = v
            filas.append(fila)
    return filas

# La forma la elige la etapa por el menor residuo de anillo, y depende del
# objeto: no se puede dar por supuesta. El CSV de Moffat existe siempre (es
# la forma de desempate); el de psfao solo si psfao ganó.
FORMA_ELEGIDA = str(PSF_MODEL.get('form', QC_C1['fit']['form_chosen'])).lower()
ES_PSFAO = (FORMA_ELEGIDA == 'psfao')
FILAS_MOFFAT = _filas_csv(SD / 'stage_e01_psf_params.csv')
_csv_psfao = SD / 'stage_e01_psfao_params.csv'
FILAS_PSFAO = _filas_csv(_csv_psfao) if _csv_psfao.exists() else []
FWHM_MED = float(np.nanmedian([r['fwhm_maj'] for r in FILAS_MOFFAT]))
FILAS = FILAS_PSFAO if ES_PSFAO else FILAS_MOFFAT
CLAVE_LAMBDA = 'lambda_A' if ES_PSFAO else 'wave_center_A'
# Los nombres van escritos porque esta celda es ANTERIOR a la copia; la §5
# comprueba que son los de `musepipe` (PSFAO_PARAM_NAMES / PSF_SHAPE_PARAMS).
NOMBRES = (list(PSF_MODEL.get('param_names',
                              ['r0', 'C', 'A', 'alpha', 'ratio', 'theta', 'beta']))
           if ES_PSFAO else
           ['y0', 'x0', 'fwhm_maj', 'fwhm_min', 'theta_deg', 'beta'])

def _cuando(p):
    p = Path(p)
    if not p.exists():
        return None, 'no está'
    t = p.stat().st_mtime
    return t, _dt.datetime.fromtimestamp(t).strftime('%Y-%m-%d %H:%M')
_t_cubo, _s_cubo = _cuando(CUBE_PATH)
_t_mod,  _s_mod  = _cuando(SD / 'psf_model.json')
print(f'  cubo (entrada) {CUBE_PATH.name:34s} {_s_cubo}')
print(f'  C1 (salida)    {"psf_model.json":34s} {_s_mod}')
C1_AL_DIA = bool(_t_cubo and _t_mod and _t_mod >= _t_cubo)
if _t_cubo and _t_mod and not C1_AL_DIA:
    print('\n   AVISO: el modelo de PSF es MÁS VIEJO que el cubo sobre el que dice')
    print('   haberse ajustado. C1 no se ha re-ejecutado desde que cambió el cubo:')
    print('   lo que auditan las §8-§12 describe una PSF que ya no es la de estos')
    print('   datos. Se arregla con:')
    print(f'      bash scripts/stage_e01_psf.sh --run-id {RUN_ID}')
print()
print('cubo      :', FORMA_CUBO, '·', WAVE_ALL.size, 'canales',
      f'({WAVE_ALL.min():.0f}-{WAVE_ALL.max():.0f} Å)')
print('primaria  :', [round(v, 2) for v in STAR_YX],
      ' compañero:', [round(v, 2) for v in COMP_YX],
      ' campo:', FIELD_YX)
print(f'máscara del compañero: r={MASK_RADIUS_PX:.2f} px'
      f' (3 × [err {float(POS["companion"].get("err_px", 0) or 0):.3f}'
      f' + fwhm prelim {float(E01.get("psf_prelim_fwhm_px", 4.0)):.1f}])')
print(f'forma elegida por la cadena: {FORMA_ELEGIDA}'
      f' — {QC_C1["model_comparison"]["reason"]}')
print(f'filas por bin: psfao {len(FILAS_PSFAO)} · moffat {len(FILAS_MOFFAT)}')
if ES_PSFAO:
    print(f'el modelo entregado es una TABLA de'
          f' {len(PSF_MODEL["param_table"]["lambda_A"])} bins que se interpola')
else:
    print('el modelo entregado son POLINOMIOS por parámetro:',
          ', '.join(f'{k}=grado {PSF_MODEL["coefficients"][k]["degree"]}'
                    for k in NOMBRES))
    print('   -> este objeto no pasa por `_evaluate_psfao`: las §8-§9 lo dicen'
          ' y miden lo que sí aplica.')


## 3 · Las funciones copiadas de `musepipe`

Aquí se invierte el corte del resto de notebooks: `_evaluate_psfao` **viaja copiada**, porque es la función bajo sospecha. `evaluate_psf_model` no viaja —para `form='psfao'` son dos líneas que delegan en ella, y copiarlo arrastraría toda la rama Moffat que este run no usa.

- `finite_values` — de `musepipe/stats.py`
- `finite_percentile` — de `musepipe/stats.py`
- `robust_sigma` — de `musepipe/stats.py`
- `VerificationError` — de `musepipe/reduction/verify.py`
- `circular_aperture_mask` — de `musepipe/reduction/verify.py`
- `extract_aperture_spectrum` — de `musepipe/reduction/verify.py`
- `TelluricError` — de `musepipe/reduction/telluric.py`
- `wavelength_axis_from_header` — de `musepipe/reduction/telluric.py`
- `MoffatFit` — de `musepipe/psf.py`
- `moffat_alpha_from_fwhm` — de `musepipe/psf.py`
- `moffat_elliptical_profile` — de `musepipe/psf.py`
- `moffat_image` — de `musepipe/psf.py`
- `fixed_radius_grid` — de `musepipe/psf.py`
- `moffat_norm` — de `musepipe/psf.py`
- `normalized_moffat_psf` — de `musepipe/psf.py`
- `source_mask` — de `musepipe/psf.py`
- `corner_background` — de `musepipe/psf.py`
- `_initial_fit_params` — de `musepipe/psf.py`
- `_pack_params` — de `musepipe/psf.py`
- `fit_moffat_image` — de `musepipe/psf.py`
- `evaluate_moffat_fit` — de `musepipe/psf.py`
- `companion_ring_metric` — de `musepipe/psf.py`
- `smooth_parameter` — de `musepipe/psf.py`
- `eval_smoothed_parameter` — de `musepipe/psf.py`
- `build_psf_model_document` — de `musepipe/psf.py`
- `_psfao_image_cached` — de `musepipe/psf.py`
- `_evaluate_psfao` — de `musepipe/psf.py`
- `evaluate_psf_model` — de `musepipe/psf.py`
- `psf_roundtrip_error` — de `musepipe/psf.py`
- `radial_hybrid_profile` — de `musepipe/psf.py`
- `evaluate_radial_profile` — de `musepipe/psf.py`
- `_bad_windows` — de `musepipe/stages/stage_e01_psfao.py`
- `make_bins` — de `musepipe/stages/stage_e01_psfao.py`
- `_ring_residual` — de `musepipe/stages/stage_e01_psfao.py`
- `_box3_apcorr` — de `musepipe/stages/stage_e01_psfao.py`
- `fit_bin` — de `musepipe/stages/stage_e01_psfao.py`
- `_psfao_param_errors` — de `musepipe/stages/stage_e01_psfao.py`
- `_psfao_fit_status` — de `musepipe/stages/stage_e01_psfao.py`
- `fit_psfao_bins` — de `musepipe/stages/stage_e01_psfao.py`
- `_norm_roundtrip` — de `musepipe/stages/stage_e01_psfao.py`
- `build_psfao_model_document` — de `musepipe/stages/stage_e01_psfao.py`
- `_good_wave_mask` — de `musepipe/stages/stage_e01_psf.py`
- `make_psf_bins` — de `musepipe/stages/stage_e01_psf.py`
- `_median_image` — de `musepipe/stages/stage_e01_psf.py`
- `_positions_from_qc` — de `musepipe/stages/stage_e01_psf.py`
- `_fit_source_mask` — de `musepipe/stages/stage_e01_psf.py`
- `_detect_core_mask` — de `musepipe/stages/stage_e01_psf.py`
- `_row_from_fit` — de `musepipe/stages/stage_e01_psf.py`
- `_moffat_fit_rows` — de `musepipe/stages/stage_e01_psf.py`
- `_apply_hybrid` — de `musepipe/stages/stage_e01_psf.py`


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from astropy.io import fits
from dataclasses import dataclass
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import least_squares
import functools
import math
import numpy as np
import warnings

MOFFAT_BETA_FLOOR = 1.05  # Moffat is only a normalizable PSF for beta > 1.
PSF_SHAPE_PARAMS = ("y0", "x0", "fwhm_maj", "fwhm_min", "theta_deg", "beta")
_PSFAO_PARAM_NAMES = ("r0", "C", "A", "alpha", "ratio", "theta", "beta")
PSFAO_PARAM_NAMES = ("r0", "C", "A", "alpha", "ratio", "theta", "beta")
DEFAULT_X0 = [0.15, 1e-4, 1.0, 0.05, 1.0, 0.0, 1.6]


def finite_values(values) -> np.ndarray:
    """Return finite values as a float64 1D array."""

    arr = np.asarray(values, dtype=np.float64)
    return arr[np.isfinite(arr)]


def finite_percentile(values, q) -> float:
    """Percentile over finite values, returning NaN for empty input."""

    vals = finite_values(values)
    if vals.size == 0:
        return np.nan
    return float(np.nanpercentile(vals, q))


def robust_sigma(values) -> float:
    """Robust 1D sigma estimate using MAD with std fallback."""

    vals = finite_values(values)
    if vals.size == 0:
        return np.nan
    med = np.nanmedian(vals)
    mad = np.nanmedian(np.abs(vals - med))
    sigma = 1.4826 * mad
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = np.nanstd(vals)
    return float(sigma)


class VerificationError(RuntimeError):
    """Raised when a requested verification cannot be completed."""


def circular_aperture_mask(shape: tuple[int, int], yx: tuple[float, float], radius: float) -> np.ndarray:
    y, x = np.indices(shape, dtype=np.float64)
    cy, cx = yx
    return (y - cy) ** 2 + (x - cx) ** 2 <= radius**2


def extract_aperture_spectrum(cube: np.ndarray, yx: tuple[float, float], radius: float) -> np.ndarray:
    mask = circular_aperture_mask(cube.shape[1:], yx, radius)
    if not mask.any():
        raise VerificationError("Aperture contains no pixels.")
    return np.nansum(cube[:, mask], axis=1)


class TelluricError(RuntimeError):
    """Raised when A3 must stop at a gate or checkpoint."""


def wavelength_axis_from_header(header: fits.Header, n_wave: int) -> np.ndarray:
    if all(key in header for key in ("CRVAL3", "CDELT3")):
        crpix = float(header.get("CRPIX3", 1.0))
        return float(header["CRVAL3"]) + (
            np.arange(int(n_wave), dtype=np.float64) + 1.0 - crpix
        ) * float(header["CDELT3"])
    if all(key in header for key in ("CRVAL3", "CD3_3")):
        crpix = float(header.get("CRPIX3", 1.0))
        return float(header["CRVAL3"]) + (
            np.arange(int(n_wave), dtype=np.float64) + 1.0 - crpix
        ) * float(header["CD3_3"])
    raise TelluricError("Could not recover wavelength axis from DATA header.")


@dataclass(frozen=True)
class MoffatFit:
    success: bool
    params: dict[str, float]
    errors: dict[str, float]
    background: float
    chi2r: float
    clip_frac: float
    n_fit: int
    message: str


def moffat_alpha_from_fwhm(fwhm, beta):
    fwhm = float(fwhm)
    beta = float(beta)
    # A Moffat has finite integral only for beta > 1; a degree-N beta(lambda)
    # polynomial from C1 can extrapolate to beta <= 0 at band edges outside its
    # fit range (seen on the LkCa 15 Moffat fit: 382/3681 channels beta<=0),
    # which sends 2**(1/beta) to an OverflowError and crashes every downstream
    # apcorr. Clamp to the physical floor: a no-op for any healthy PSF (beta>1),
    # and it keeps the aperture correction finite where the model is being
    # extrapolated into the non-normalizable regime.
    if not math.isfinite(beta) or beta < MOFFAT_BETA_FLOOR:
        beta = MOFFAT_BETA_FLOOR
    denom = 2.0 * math.sqrt(max(2.0 ** (1.0 / beta) - 1.0, 1e-12))
    return fwhm / denom


def moffat_elliptical_profile(dy, dx, fwhm_maj, fwhm_min, theta_deg, beta):
    """Unit-peak elliptical Moffat profile."""

    dy = np.asarray(dy, dtype=np.float64)
    dx = np.asarray(dx, dtype=np.float64)
    theta = np.deg2rad(float(theta_deg))
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)
    x_rot = dx * cos_t + dy * sin_t
    y_rot = -dx * sin_t + dy * cos_t
    alpha_maj = moffat_alpha_from_fwhm(fwhm_maj, beta)
    alpha_min = moffat_alpha_from_fwhm(fwhm_min, beta)
    rr = (x_rot / alpha_maj) ** 2 + (y_rot / alpha_min) ** 2
    return (1.0 + rr) ** (-float(beta))


def moffat_image(shape, y0, x0, fwhm_maj, fwhm_min, theta_deg, beta, amplitude=1.0, background=0.0):
    yy, xx = np.indices(shape, dtype=np.float64)
    profile = moffat_elliptical_profile(
        yy - float(y0),
        xx - float(x0),
        fwhm_maj,
        fwhm_min,
        theta_deg,
        beta,
    )
    return float(background) + float(amplitude) * profile


def fixed_radius_grid(norm_radius_px):
    radius = float(norm_radius_px)
    half = int(math.ceil(radius))
    yy, xx = np.mgrid[-half : half + 1, -half : half + 1].astype(np.float64)
    mask = (yy**2 + xx**2) <= radius**2
    return yy, xx, mask


def moffat_norm(params, norm_radius_px=25.0):
    yy, xx, mask = fixed_radius_grid(norm_radius_px)
    profile = moffat_elliptical_profile(
        yy,
        xx,
        params["fwhm_maj"],
        params["fwhm_min"],
        params.get("theta_deg", 0.0),
        params["beta"],
    )
    norm = float(np.nansum(profile[mask]))
    if not np.isfinite(norm) or norm <= 0:
        raise RuntimeError("Invalid Moffat normalization.")
    return norm


def normalized_moffat_psf(dy, dx, params, norm_radius_px=25.0):
    profile = moffat_elliptical_profile(
        dy,
        dx,
        params["fwhm_maj"],
        params["fwhm_min"],
        params.get("theta_deg", 0.0),
        params["beta"],
    )
    return profile / moffat_norm(params, norm_radius_px=norm_radius_px)


def source_mask(shape, centers_yx, radius_px):
    yy, xx = np.indices(shape, dtype=np.float64)
    mask = np.zeros(shape, dtype=bool)
    for center in centers_yx or ():
        if center is None:
            continue
        y, x = map(float, center)
        mask |= (yy - y) ** 2 + (xx - x) ** 2 <= float(radius_px) ** 2
    return mask


def corner_background(image, corner_size=12):
    img = np.asarray(image, dtype=np.float64)
    c = int(min(corner_size, max(1, img.shape[0] // 4), max(1, img.shape[1] // 4)))
    vals = np.concatenate(
        [
            img[:c, :c].ravel(),
            img[:c, -c:].ravel(),
            img[-c:, :c].ravel(),
            img[-c:, -c:].ravel(),
        ]
    )
    vals = vals[np.isfinite(vals)]
    return float(np.nanmedian(vals)) if vals.size else 0.0


def _initial_fit_params(image, center_yx, background, fit_mask):
    img = np.asarray(image, dtype=np.float64)
    y0, x0 = map(float, center_yx)
    peak_region = fit_mask & np.isfinite(img)
    peak = float(np.nanmax(img[peak_region] - float(background))) if np.any(peak_region) else 1.0
    if not np.isfinite(peak) or peak <= 0:
        peak = 1.0
    return np.array([peak, y0, x0, 4.0, 4.0, 0.0, 2.5], dtype=np.float64)


def _pack_params(values):
    amp, y0, x0, fmaj, fmin, theta, beta = values
    if fmin > fmaj:
        fmaj, fmin = fmin, fmaj
        theta += 90.0
    theta = ((float(theta) + 90.0) % 180.0) - 90.0
    return {
        "amplitude": float(amp),
        "y0": float(y0),
        "x0": float(x0),
        "fwhm_maj": float(fmaj),
        "fwhm_min": float(fmin),
        "theta_deg": theta,
        "beta": float(beta),
    }


def fit_moffat_image(
    image,
    *,
    center_yx,
    fit_radius_px=28.0,
    mask=None,
    background=None,
    core_mask_px=0.0,
    sigma_clip=3.0,
    max_iter=3,
    min_pixels=40,
):
    """Fit a fixed-background elliptical Moffat image model."""

    img = np.asarray(image, dtype=np.float64)
    if img.ndim != 2:
        raise ValueError(f"Expected a 2D image, got {img.shape}.")
    ny, nx = img.shape
    cy, cx = map(float, center_yx)
    yy, xx = np.indices(img.shape, dtype=np.float64)
    rr = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    fit_mask = (rr <= float(fit_radius_px)) & np.isfinite(img)
    if mask is not None:
        fit_mask &= ~np.asarray(mask, dtype=bool)
    if float(core_mask_px) > 0:
        fit_mask &= rr > float(core_mask_px)
    if int(np.count_nonzero(fit_mask)) < int(min_pixels):
        raise RuntimeError("Too few pixels for Moffat fit.")

    background = corner_background(img) if background is None else float(background)
    y = img[fit_mask] - background
    ypix = yy[fit_mask]
    xpix = xx[fit_mask]
    good = np.ones(y.size, dtype=bool)
    x0 = _initial_fit_params(img, center_yx, background, fit_mask)
    lower = [0.0, cy - 3.0, cx - 3.0, 0.6, 0.6, -90.0, 1.05]
    upper = [
        max(float(np.nanmax(y)) * 3.0, 1.0),
        cy + 3.0,
        cx + 3.0,
        max(2.0, fit_radius_px),
        max(2.0, fit_radius_px),
        90.0,
        12.0,
    ]

    fit = None
    for _ in range(int(max_iter)):
        if int(np.count_nonzero(good)) < int(min_pixels):
            break

        def resid(values):
            params = _pack_params(values)
            model = params["amplitude"] * moffat_elliptical_profile(
                ypix[good] - params["y0"],
                xpix[good] - params["x0"],
                params["fwhm_maj"],
                params["fwhm_min"],
                params["theta_deg"],
                params["beta"],
            )
            return model - y[good]

        fit = least_squares(resid, x0=x0, bounds=(lower, upper), max_nfev=500)
        full_params = _pack_params(fit.x)
        model_all = full_params["amplitude"] * moffat_elliptical_profile(
            ypix - full_params["y0"],
            xpix - full_params["x0"],
            full_params["fwhm_maj"],
            full_params["fwhm_min"],
            full_params["theta_deg"],
            full_params["beta"],
        )
        residual_all = model_all - y
        sigma = robust_sigma(residual_all[good])
        if sigma_clip is None or not np.isfinite(sigma) or sigma <= 0:
            break
        new_good = np.abs(residual_all) <= float(sigma_clip) * sigma
        if np.array_equal(new_good, good):
            break
        good = new_good
        x0 = fit.x

    if fit is None:
        raise RuntimeError("Moffat fit did not run.")
    params = _pack_params(fit.x)
    model = params["amplitude"] * moffat_elliptical_profile(
        ypix[good] - params["y0"],
        xpix[good] - params["x0"],
        params["fwhm_maj"],
        params["fwhm_min"],
        params["theta_deg"],
        params["beta"],
    )
    resid = model - y[good]
    sigma = robust_sigma(resid)
    dof = max(1, int(np.count_nonzero(good)) - 7)
    chi2r = float(np.nansum((resid / sigma) ** 2) / dof) if np.isfinite(sigma) and sigma > 0 else np.nan
    errors = {key: np.nan for key in ("amplitude",) + PSF_SHAPE_PARAMS}
    if fit.jac is not None and fit.jac.size and np.isfinite(sigma) and sigma > 0:
        try:
            cov = np.linalg.pinv(fit.jac.T @ fit.jac) * sigma**2
            err_values = np.sqrt(np.clip(np.diag(cov), 0.0, np.inf))
            for key, err in zip(("amplitude", "y0", "x0", "fwhm_maj", "fwhm_min", "theta_deg", "beta"), err_values):
                errors[key] = float(err)
        except Exception:
            pass

    return MoffatFit(
        success=bool(fit.success),
        params=params,
        errors=errors,
        background=background,
        chi2r=chi2r,
        clip_frac=float(1.0 - np.count_nonzero(good) / y.size),
        n_fit=int(np.count_nonzero(good)),
        message=str(fit.message),
    )


def evaluate_moffat_fit(shape, fit: MoffatFit):
    p = fit.params
    return moffat_image(
        shape,
        p["y0"],
        p["x0"],
        p["fwhm_maj"],
        p["fwhm_min"],
        p["theta_deg"],
        p["beta"],
        amplitude=p["amplitude"],
        background=fit.background,
    )


def companion_ring_metric(image, model, primary_yx, companion_yx, *, width_px=3.0, source_exclusion_radius_px=0.0):
    img = np.asarray(image, dtype=np.float64)
    mod = np.asarray(model, dtype=np.float64)
    yy, xx = np.indices(img.shape, dtype=np.float64)
    py, px = map(float, primary_yx)
    cy, cx = map(float, companion_yx)
    radius = math.hypot(cy - py, cx - px)
    rr = np.sqrt((yy - py) ** 2 + (xx - px) ** 2)
    ann = np.abs(rr - radius) <= float(width_px) / 2.0
    if source_exclusion_radius_px and source_exclusion_radius_px > 0:
        ann &= (yy - cy) ** 2 + (xx - cx) ** 2 > float(source_exclusion_radius_px) ** 2
    halo = np.abs(mod)
    vals = np.abs(img - mod) / np.maximum(halo, np.nanmedian(halo[ann]) * 0.05)
    vals = vals[ann & np.isfinite(vals)]
    if vals.size == 0:
        return {"radius_px": float(radius), "median_pct": np.nan, "p90_pct": np.nan}
    return {
        "radius_px": float(radius),
        "median_pct": float(100.0 * np.nanmedian(vals)),
        "p90_pct": float(100.0 * finite_percentile(vals, 90.0)),
    }


def smooth_parameter(wavelengths_A, values, *, max_degree=2, wave_ref_A=None, wave_scale_A=1000.0):
    wave = np.asarray(wavelengths_A, dtype=np.float64)
    vals = np.asarray(values, dtype=np.float64)
    good = np.isfinite(wave) & np.isfinite(vals)
    if int(np.count_nonzero(good)) == 0:
        raise ValueError("No finite values to smooth.")
    wave_ref = float(np.nanmedian(wave[good])) if wave_ref_A is None else float(wave_ref_A)
    x = (wave[good] - wave_ref) / float(wave_scale_A)
    y = vals[good]
    best = None
    for deg in range(0, min(int(max_degree), y.size - 1) + 1):
        coeff_high = np.polyfit(x, y, deg)
        pred = np.polyval(coeff_high, x)
        rss = float(np.nansum((y - pred) ** 2))
        k = deg + 1
        aic = y.size * math.log(max(rss / max(y.size, 1), 1e-24)) + 2 * k
        if best is None or aic < best["aic"]:
            best = {"degree": deg, "coeff_high": coeff_high, "rss": rss, "aic": aic}
    coeff_low = best["coeff_high"][::-1].astype(float).tolist()
    return {
        "degree": int(best["degree"]),
        "coefficients": coeff_low,
        "wave_ref_A": wave_ref,
        "wave_scale_A": float(wave_scale_A),
        "model": "polynomial",
    }


def eval_smoothed_parameter(spec, wavelength_A):
    x = (float(wavelength_A) - float(spec["wave_ref_A"])) / float(spec.get("wave_scale_A", 1000.0))
    coeff = np.asarray(spec["coefficients"], dtype=np.float64)
    return float(np.polynomial.polynomial.polyval(x, coeff))


def build_psf_model_document(
    wavelength_bins_A,
    fit_rows,
    *,
    form="moffat",
    norm_radius_px=25.0,
    hybrid=False,
):
    waves = np.asarray(wavelength_bins_A, dtype=np.float64)
    smoothing = {}
    for key in PSF_SHAPE_PARAMS:
        smoothing[key] = smooth_parameter(waves, [row[key] for row in fit_rows])
    return {
        "form": str(form),
        "norm_radius_px": float(norm_radius_px),
        "coefficients": smoothing,
        "hybrid": bool(hybrid),
    }


@functools.lru_cache(maxsize=16384)
def _psfao_image_cached(x_key, npix, system_name, samp, norm_radius):
    """Build (and cache) the normalised Psfao image for one parameter set.

    Building the Psfao model is an FFT (~6 ms). During per-channel PSF fitting
    (C3/C4) the optimiser evaluates the SAME wavelength/params many times while
    varying only flux/position, so caching the image (keyed on the params, grid
    size, sampling and norm radius) turns hours into minutes. Returns the even
    image, its norm_radius integral, and the grid centre."""

    from maoppy.instrument import muse_nfm, muse_wfm
    from maoppy.psfmodel import Psfao

    system = muse_wfm if str(system_name).lower().endswith("wfm") else muse_nfm
    model = Psfao((npix, npix), system=system, samp=samp)
    # Clip to Psfao's physical bounds (smoothed/interpolated params can drift out
    # of range at edge/gap wavelengths).
    low, high = model.bounds
    eps = 1e-6
    x = [
        float(np.clip(
            xi,
            low[i] + eps if np.isfinite(low[i]) else -np.inf,
            high[i] - eps if np.isfinite(high[i]) else np.inf,
        ))
        for i, xi in enumerate(x_key)
    ]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        img = np.asarray(model(x), dtype=np.float64)  # peak at (npix//2, npix//2)
    c = npix // 2
    gy, gx = np.mgrid[0:npix, 0:npix]
    rr = np.hypot(gy - c, gx - c)
    total = float(np.nansum(img[rr <= float(norm_radius)]))
    if not np.isfinite(total) or total <= 0:
        raise RuntimeError("Psfao normalization within norm_radius failed.")
    return img, total, c


def _evaluate_psfao(model_doc, wavelength_A, dy, dx):
    """Evaluate a physical AO PSF (maoppy Psfao) on the (dy, dx) offsets,
    normalised so it sums to 1 within ``norm_radius_px``. Mirrors the Moffat
    branch's contract so aperture-correction/growth-curve/optimal/psffit
    consumers are agnostic to the PSF form. Params come from the C1 per-bin
    ``param_table`` (interpolated) or ``smoothed_poly``. The expensive FFT build
    is cached in ``_psfao_image_cached``; here we only re-sample it."""

    from maoppy.instrument import muse_nfm
    from scipy.ndimage import map_coordinates

    dy = np.asarray(dy, dtype=np.float64)
    dx = np.asarray(dx, dtype=np.float64)
    if dy.shape != dx.shape:
        raise ValueError("psfao evaluation expects matching dy/dx offset arrays.")
    # FWHM perturbation (C3/E4 sensitivity tests). The Psfao parameters live in
    # the PSD, so there is no coefficient to multiply the way the Moffat branch
    # does: the geometric equivalent is to sample the built PSF on offsets
    # divided by the scale (a dilation by `scale`), with 1/scale**2 conserving
    # the integral. See `scaled_psf_model`.
    fwhm_scale = float(model_doc.get("psf_fwhm_scale", 1.0))
    if not np.isfinite(fwhm_scale) or fwhm_scale <= 0:
        raise ValueError(f"psf_fwhm_scale must be finite and > 0, got {fwhm_scale!r}.")
    if fwhm_scale != 1.0:
        dy = dy / fwhm_scale
        dx = dx / fwhm_scale
    names = model_doc.get("param_names", _PSFAO_PARAM_NAMES)
    # Snap the wavelength to a coarse bin before building the PSF: C1 fits the
    # Psfao parameters in 100 A bins and the PSF varies <0.5% within ~50 A, so
    # binning lets consecutive channels share ONE cached FFT build (3681 builds
    # -> ~90) with negligible loss. Sampling still uses the exact per-call
    # offsets, so per-channel positions/flux stay exact.
    wave_bin = float(model_doc.get("psfao_wave_bin_A", 50.0))
    w_eff = round(float(wavelength_A) / wave_bin) * wave_bin if wave_bin > 0 else float(wavelength_A)
    table = model_doc.get("param_table")
    if table:
        # Interpolate the per-bin fits (Psfao PSD params are degenerate, so
        # smoothing them independently then reconstructing corrupts the PSF).
        lam = np.asarray(table["lambda_A"], dtype=np.float64)
        w = float(np.clip(w_eff, lam.min(), lam.max()))
        x = [float(np.interp(w, lam, np.asarray(table[name], dtype=np.float64))) for name in names]
    else:
        poly = model_doc.get("smoothed_poly") or {}
        x = [float(np.polyval(np.asarray(poly[name], dtype=np.float64), w_eff)) for name in names]
    system_name = "muse_wfm" if str(model_doc.get("system", "muse_nfm")).lower().endswith("wfm") else "muse_nfm"
    samp = float(muse_nfm.samp(w_eff * 1e-10))
    norm_radius = float(model_doc.get("norm_radius_px", 25.0))
    # Grid sizing: two regimes, both giving a source-position-INDEPENDENT npix so
    # the cache is reused across the star/companion/control evaluations at a given
    # wavelength.
    #   - apcorr/growth-curve callers pass offsets within norm_radius -> npix from
    #     norm_radius (small, fast).
    #   - psffit evaluates over the full image (offsets up to ~image size); the PSF
    #     is only needed over the joint fit region (source separation + fit radius),
    #     so use a fixed grid_reach (default 100 px, config `psfao_grid_reach_px`).
    #     This captures the star halo at the companion (~71 px) while avoiding the
    #     ~316 px FFTs the full-image offsets would otherwise force; offsets beyond
    #     the grid sample as 0 (negligible PSF there).
    max_off = 0.0
    if dy.size:
        max_off = max(float(np.nanmax(np.abs(dy))), float(np.nanmax(np.abs(dx))))
    if max_off <= norm_radius:
        reach = norm_radius
    else:
        # 140 px is where the PSF-fit companion flux converges for this geometry
        # (100 under-samples the AO halo → ~3% high; 140/180/250 agree to 0.2%).
        reach = max(norm_radius, float(model_doc.get("psfao_grid_reach_px", 140.0)))
    npix = 2 * (int(np.ceil(reach)) + 2)  # even, source at npix//2
    img, total, c = _psfao_image_cached(
        tuple(round(v, 10) for v in x), npix, system_name, round(samp, 10), round(norm_radius, 6)
    )
    rows = (c + dy).ravel()
    cols = (c + dx).ravel()
    vals = map_coordinates(img, [rows, cols], order=1, mode="constant", cval=0.0).reshape(dy.shape)
    return vals / (total * fwhm_scale ** 2)


def evaluate_psf_model(model_doc, wavelength_A, dy, dx):
    form = str(model_doc.get("form", "moffat")).lower()
    if form == "psfao":
        return _evaluate_psfao(model_doc, wavelength_A, dy, dx)
    if form != "moffat":
        raise ValueError(f"Unsupported psf_model form={form!r}; expected 'moffat' or 'psfao'.")
    params = {
        key: eval_smoothed_parameter(model_doc["coefficients"][key], wavelength_A)
        for key in PSF_SHAPE_PARAMS
    }
    return normalized_moffat_psf(
        dy,
        dx,
        params,
        norm_radius_px=float(model_doc.get("norm_radius_px", 25.0)),
    )


def psf_roundtrip_error(model_doc, wavelengths_A):
    yy, xx, mask = fixed_radius_grid(float(model_doc.get("norm_radius_px", 25.0)))
    errors = []
    for wave in wavelengths_A:
        vals = evaluate_psf_model(model_doc, float(wave), yy, xx)
        errors.append(abs(float(np.nansum(vals[mask])) - 1.0))
    return float(np.nanmax(errors)) if errors else np.nan


def radial_hybrid_profile(residual, center_yx, *, mask=None, bin_width_px=1.0, smoothing_scale_px=4.0):
    resid = np.asarray(residual, dtype=np.float64)
    yy, xx = np.indices(resid.shape, dtype=np.float64)
    rr = np.sqrt((yy - float(center_yx[0])) ** 2 + (xx - float(center_yx[1])) ** 2)
    valid = np.isfinite(resid)
    if mask is not None:
        valid &= ~np.asarray(mask, dtype=bool)
    bins = np.floor(rr / float(bin_width_px)).astype(int)
    nbin = int(np.nanmax(bins)) + 1
    profile = np.full(nbin, np.nan, dtype=np.float64)
    radii = (np.arange(nbin, dtype=np.float64) + 0.5) * float(bin_width_px)
    for b in range(nbin):
        pix = valid & (bins == b)
        if np.count_nonzero(pix) >= 3:
            profile[b] = np.nanmedian(resid[pix])
    finite = np.isfinite(profile)
    if np.count_nonzero(finite) >= 2:
        profile[~finite] = np.interp(radii[~finite], radii[finite], profile[finite])
    else:
        profile[~finite] = 0.0
    sigma_bins = max(float(smoothing_scale_px) / float(bin_width_px), 0.0)
    if sigma_bins > 0:
        profile = gaussian_filter1d(profile, sigma=sigma_bins, mode="nearest")
    return radii, profile


def evaluate_radial_profile(shape, center_yx, radii, profile):
    yy, xx = np.indices(shape, dtype=np.float64)
    rr = np.sqrt((yy - float(center_yx[0])) ** 2 + (xx - float(center_yx[1])) ** 2)
    return np.interp(rr.ravel(), np.asarray(radii), np.asarray(profile), left=profile[0], right=profile[-1]).reshape(shape)


def _bad_windows(cfg):
    if cfg.get("drop_wave_min_A") is not None and cfg.get("drop_wave_max_A") is not None:
        return [[float(cfg["drop_wave_min_A"]), float(cfg["drop_wave_max_A"])]]
    return [[5780.0, 6050.0]]


def make_bins(wave, bin_A, bad_windows, min_channels=3):
    lo, hi = float(wave.min()), float(wave.max())
    edges = np.arange(lo, hi + bin_A, bin_A)
    bins = []
    for a, b in zip(edges[:-1], edges[1:]):
        mid = 0.5 * (a + b)
        if any(w0 <= mid <= w1 for w0, w1 in bad_windows):
            continue
        sel = (wave >= a) & (wave < b)
        if sel.sum() >= min_channels:
            bins.append((a, b, mid, sel))
    return bins


def _ring_residual(image, model, companion_yx, mask_radius, width=1.5):
    ny, nx = image.shape
    cy, cx = ny // 2, nx // 2
    yy, xx = np.mgrid[0:ny, 0:nx]
    r = np.hypot(yy - cy, xx - cx)
    comp_r = float(np.hypot(companion_yx[0] - cy, companion_yx[1] - cx))
    comp = np.hypot(yy - companion_yx[0], xx - companion_yx[1])
    ann = (np.abs(r - comp_r) < width) & np.isfinite(image) & (comp > mask_radius)
    halo = np.nanmedian(image[ann])
    if not np.isfinite(halo) or halo == 0:
        return float("nan")
    return float(100.0 * np.nanmedian(np.abs(image[ann] - model[ann])) / halo)


def _box3_apcorr(system, lam_A, params, norm_radius):
    """Box3 aperture correction from one Psfao parameter set (outlier metric)."""
    from maoppy.instrument import muse_nfm
    from maoppy.psfmodel import Psfao
    try:
        m = Psfao((52, 52), system=system, samp=float(muse_nfm.samp(float(lam_A) * 1e-10)))
        low, high = m.bounds
        eps = 1e-6
        x = [float(np.clip(float(v),
                           low[i] + eps if np.isfinite(low[i]) else -np.inf,
                           high[i] - eps if np.isfinite(high[i]) else np.inf))
             for i, v in enumerate(params)]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            img = np.asarray(m(x), dtype=np.float64)
        c = 26
        yy, xx = np.mgrid[0:52, 0:52]
        r = np.hypot(yy - c, xx - c)
        total = float(np.nansum(img[r <= norm_radius]))
        box = float(np.nansum(img[(np.abs(yy - c) <= 1) & (np.abs(xx - c) <= 1)]))
        return float(total / box) if box > 0 else np.nan
    except Exception:
        return np.nan


def fit_bin(image, var, samp, system, companion_yx, mask_radius, fit_radius, x0, field_yx=None):
    """Ajuste Psfao de un bin.

    `field_yx` enmascara una fuente de campo igual que hace la rama Moffat. Sin
    esto las dos formas se ajustaban con MASCARAS DISTINTAS —Moffat con
    companero + fuente de campo, psfao solo con el companero— y
    `model_comparison` dejaba de comparar peras con peras: el sesgo iba siempre
    contra psfao, porque era la unica que se comia el contaminante. Se vio en
    ROXs 42B b, donde enmascarar ROXs 42B cc1 mejoro Moffat (8.38 -> 6.88%) y
    dejo psfao intacta (9.80 -> 10.69%).
    """

    from maoppy.psfmodel import Psfao
    from maoppy.psffit import psffit

    ny, nx = image.shape
    cy, cx = ny // 2, nx // 2
    yy, xx = np.mgrid[0:ny, 0:nx]
    r = np.hypot(yy - cy, xx - cx)
    comp = np.hypot(yy - companion_yx[0], xx - companion_yx[1])
    mask = np.isfinite(image) & (comp > mask_radius) & (r < fit_radius)
    if field_yx is not None:
        mask &= np.hypot(yy - float(field_yx[0]), xx - float(field_yx[1])) > mask_radius
    weights = np.where(mask, 1.0 / np.clip(var, 1e-6, None), 0.0)
    imgf = np.where(np.isfinite(image), image, 0.0)
    model = Psfao((ny, nx), system=system, samp=float(samp))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res = psffit(imgf, model, x0, weights=weights, flux_bck=(True, True), max_nfev=400)
    amp, bck = res.flux_bck
    recon = amp * model(res.x, dx=res.dxdy[0], dy=res.dxdy[1]) + bck
    ring = _ring_residual(image, recon, companion_yx, mask_radius)
    unchanged = bool(
        np.allclose(np.asarray(res.x, dtype=float), np.asarray(x0, dtype=float), rtol=0.0, atol=1e-8)
        and np.allclose(np.asarray(res.dxdy, dtype=float), 0.0, rtol=0.0, atol=1e-8)
    )
    # An exact solution may legitimately equal x0. Treat it as a stall only when
    # the optimizer also stopped immediately without exploring the parameter space.
    stalled = bool(unchanged and int(res.nfev) <= 2)
    optimizer = {
        "success": bool(res.success),
        "status": int(res.status),
        "message": str(res.message),
        "nfev": int(res.nfev),
        "cost": float(res.cost),
        "stalled_at_initial": stalled,
    }
    errors = _psfao_param_errors(res)
    return (list(map(float, res.x)), float(amp), float(bck), tuple(map(float, res.dxdy)),
            ring, recon, optimizer, errors)


def _psfao_param_errors(res):
    """Las incertidumbres formales del ajuste, `{param: sigma}`.

    `psffit` ya las calcula —`1/sqrt(diag(JtJ))`, al final de `maoppy/psffit.py`—
    y las deja en `res.x_std` / `res.dxdy_std`, pero hasta ahora se tiraban: el
    CSV de Moffat traia sus siete columnas `*_err` y el de psfao ninguna, asi que
    no habia forma de saber si un parametro que salta entre bins vecinos esta
    medido o no.

    OJO con lo que significan. Los pesos del ajuste son `1/STAT`, y
    `docs/noise_model.md` tiene medido que STAT subestima el ruido, asi que esto
    es una **cota inferior formal**, no la incertidumbre real; ademas Moffat usa
    otra convencion (`pinv(JtJ) * sigma^2` con sigma empirica del residuo, en
    `psf.py:fit_moffat_image`). Sirven para ver la estabilidad DENTRO de cada
    forma, no para comparar tamanos ENTRE formas.

    Un parametro pegado a su limite fisico tiene gradiente nulo y su entrada de
    la diagonal sale 0 -> `1/sqrt(0)` es infinito. Eso no es una incertidumbre
    infinita, es «no medido»: se devuelve NaN, que es lo que los graficos y las
    estadisticas robustas saben ignorar.
    """

    def _limpio(value):
        v = float(value)
        return v if np.isfinite(v) and v > 0 else float("nan")

    nombres = [f"{n}_err" for n in PSFAO_PARAM_NAMES]
    try:
        x_std = np.asarray(res.x_std, dtype=float).ravel()
        dxdy_std = np.asarray(res.dxdy_std, dtype=float).ravel()
    except (AttributeError, TypeError, ValueError):  # pragma: no cover - defensive
        return {name: float("nan") for name in (*nombres, "dx_err", "dy_err")}
    errors = {name: (_limpio(x_std[i]) if i < x_std.size else float("nan"))
              for i, name in enumerate(nombres)}
    # `res.dxdy` es (dx, dy), y `res.dxdy_std` va en el mismo orden.
    errors["dx_err"] = _limpio(dxdy_std[0]) if dxdy_std.size > 0 else float("nan")
    errors["dy_err"] = _limpio(dxdy_std[1]) if dxdy_std.size > 1 else float("nan")
    return errors


def _psfao_fit_status(optimizer):
    """El veredicto de un intento: `ok`, estancado en el arranque, o fallido."""

    if optimizer["stalled_at_initial"]:
        return "fit_stalled:initial_vector"
    if not optimizer["success"]:
        return f"fit_failed:optimizer_status_{optimizer['status']}"
    return "ok"


def fit_psfao_bins(cube, stat, wave, bins, system, companion, mask_radius, fit_radius, *,
                   x0=None, field_yx=None, warm_start=True):
    """Fit the Psfao model per wavelength bin (companion masked).

    Returns ``(rows, recons)`` where ``rows`` is the per-bin parameter table
    (identical structure to the legacy loop) and ``recons`` maps each bin centre
    wavelength to ``(bin_image, reconstructed_model)`` so a caller can score the
    reconstruction with any ring metric it likes.

    ``warm_start`` fits every bin from its neighbours as well as from the shared
    ``x0``, and keeps the attempt with the LOWEST optimiser cost — the weighted
    least squares of that same bin, so the attempts are directly comparable. A
    bin only changes if another start vector fits its own data strictly better;
    where ``x0`` already wins, the parameters are unchanged bit for bit. Two
    passes, both deterministic: forward from the last accepted bin, then
    backward from the next one, so a good solution propagates in either
    direction. With ``warm_start=False`` each bin gets the single ``x0`` fit.

    This matters because every bin left out is a HOLE in the ``param_table``
    that ``_evaluate_psfao`` then spans with a straight line, and that is where
    the plateau/step structure C2-C4 inherit through the growth curve is born.
    Rescuing only the stalled bins is not enough: in ROXs 12 b the 8100-8300 A
    bins CONVERGED, from ``x0``, onto a second minimum 8x worse in cost
    (7.5e5 vs 9.4e4) whose box3 aperture correction fell 10% off the chromatic
    trend, so the outlier guard in ``build_psfao_model_document`` dropped them —
    and, being converged, they seeded the warm start that then stalled the three
    bins above them. Fitting them from either neighbour recovers the good
    minimum in all six (cost 8.7-9.4e4, ring residual better in all of them,
    box3 back on the trend), which closes the 700 A gap."""

    from maoppy.instrument import muse_nfm

    x0 = DEFAULT_X0 if x0 is None else x0

    def _intento(datos, arranque, etiqueta):
        """Un ajuste del bin, con su veredicto. Propaga lo que reviente."""

        img, var, samp, mid = datos
        params, amp, bck, dxdy, ring, recon, optimizer, errors = fit_bin(
            img, var, samp, system, companion, mask_radius, fit_radius,
            list(arranque), field_yx=field_yx)
        estado = _psfao_fit_status(optimizer)
        row = {"lambda_A": float(mid), "samp": samp, "amp": amp, "bck": bck,
               "dy": dxdy[1], "dx": dxdy[0], "ring_residual_pct": ring,
               "optimizer_success": optimizer["success"],
               "optimizer_status": optimizer["status"],
               "optimizer_message": optimizer["message"],
               "optimizer_nfev": optimizer["nfev"],
               "optimizer_cost": optimizer["cost"],
               "optimizer_stalled_at_initial": optimizer["stalled_at_initial"],
               "start_vector": etiqueta,
               "status": estado}
        row.update({name: params[i] for i, name in enumerate(PSFAO_PARAM_NAMES)})
        # Las incertidumbres formales del intento que se queda. NaN donde el
        # parametro esta pegado a su limite fisico.
        row.update(errors)
        return {"row": row, "params": list(params), "recon": recon,
                "ok": estado == "ok", "cost": float(optimizer["cost"])}

    def _intento_suave(datos, arranque, etiqueta):
        try:
            return _intento(datos, arranque, etiqueta)
        except Exception:  # pragma: no cover - defensive
            return None

    def _mejor(actual, nuevo):
        """El intento que se queda: `ok` gana a no-`ok`, y entre dos `ok`, el de
        menor coste. El empate lo gana el que ya estaba, para no mover un bin
        que nadie mejora."""

        if nuevo is None:
            return actual
        if actual is None:
            return nuevo
        if nuevo["ok"] != actual["ok"]:
            return nuevo if nuevo["ok"] else actual
        if nuevo["ok"] and nuevo["cost"] < actual["cost"]:
            return nuevo
        return actual

    datos = []
    for (a, b, mid, sel) in bins:
        datos.append((np.nanmedian(cube[sel], axis=0),
                      np.nanmedian(stat[sel], axis=0),
                      float(muse_nfm.samp(mid * 1e-10)),
                      float(mid)))

    aceptado = [None] * len(bins)
    reventados = {}
    x0_caliente = None   # el ultimo bin aceptado como bueno, hacia el rojo
    for i, d in enumerate(datos):
        try:
            mejor = _intento(d, x0, "initial_vector")
        except Exception as exc:  # pragma: no cover - defensive
            reventados[i] = {"lambda_A": d[3], "status": f"fit_failed:{exc}"}
            continue
        if warm_start and x0_caliente is not None:
            mejor = _mejor(mejor, _intento_suave(d, x0_caliente, "warm_start"))
        aceptado[i] = mejor
        if mejor["ok"]:
            x0_caliente = mejor["params"]

    # Segunda pasada, hacia el azul: un bin cuyo unico vecino bueno esta al rojo
    # —el caso de 8400-8600 A en ROXs 12 b— no tiene de donde partir en la
    # primera. Se propaga el ajuste ya aceptado del bin siguiente.
    if warm_start:
        for i in range(len(datos) - 2, -1, -1):
            siguiente = aceptado[i + 1]
            if aceptado[i] is None or siguiente is None or not siguiente["ok"]:
                continue
            aceptado[i] = _mejor(aceptado[i],
                                 _intento_suave(datos[i], siguiente["params"], "warm_start_back"))

    rows, recons = [], {}
    for i, d in enumerate(datos):
        if i in reventados:
            rows.append(reventados[i])
            continue
        acc = aceptado[i]
        rows.append(acc["row"])
        if acc["ok"]:
            recons[d[3]] = (d[0], acc["recon"])
    return rows, recons


def _norm_roundtrip(system, lam_A, poly, norm_radius):
    from maoppy.instrument import muse_nfm
    from maoppy.psfmodel import Psfao
    try:
        x = [float(np.polyval(poly[n], lam_A)) for n in PSFAO_PARAM_NAMES]
        npix = int(4 * norm_radius)
        samp = float(muse_nfm.samp(lam_A * 1e-10))
        m = Psfao((npix, npix), system=system, samp=samp)
        img = m(x)
        cy, cx = npix // 2, npix // 2
        yy, xx = np.mgrid[0:npix, 0:npix]
        inside = np.hypot(yy - cy, xx - cx) <= norm_radius
        total = float(np.nansum(img[inside]))
        return float(abs(total / np.nansum(img) - 1.0)) if np.nansum(img) else float("nan")
    except Exception:
        return float("nan")


def build_psfao_model_document(rows, system, norm_radius, fit_radius, *, system_name="muse_nfm",
                               wave_bin_A=None):
    """Assemble the psfao ``psf_model.json`` document from per-bin fit rows.

    Split out of ``run_stage_e01_psfao`` (behaviour unchanged) so the canonical
    stage can build the winning-form document when Psfao is selected. Returns the
    psf_model dict plus a small ``meta`` dict for QC assembly.

    ``wave_bin_A`` is the grid ``_evaluate_psfao`` snaps its wavelength to. It
    travels in the document because that is where the consumer reads it, and it
    defaults to the width of the bins actually fitted here: the parameters exist
    only at those wavelengths, so anything finer interpolates the degenerate PSD
    parameters between two bins and lands outside the valley (measured: a 1%
    error in the halo becomes 13% in the psffit amplitude). ``None`` leaves the
    key out, and ``psf.py`` then falls back to its historical 50 A."""

    ok = [r for r in rows if r.get("status") == "ok" and np.isfinite(r.get("ring_residual_pct", np.nan))]
    lam = np.array([r["lambda_A"] for r in ok])
    poly = {}
    for name in PSFAO_PARAM_NAMES:
        vals = np.array([r[name] for r in ok])
        deg = 2 if len(vals) >= 3 else 1
        poly[name] = list(map(float, np.polyfit(lam, vals, deg))) if len(vals) else []

    # Robust outlier rejection on the per-bin box3 aperture fraction: Psfao PSD
    # params are degenerate, so a failed bin shows up as an off-trend growth
    # curve rather than an obvious single-param spike. Keep the surviving per-bin
    # fits as an interpolation table (smoothing params directly corrupts the PSF).
    box3 = np.array([_box3_apcorr(system, float(r["lambda_A"]), [r[n] for n in PSFAO_PARAM_NAMES], norm_radius)
                     for r in ok])
    # The box3 growth curve spans a wide but smooth range (~20-130), so a global
    # MAD misses local failures; reject bins whose log-apcorr deviates from the
    # smooth wavelength trend (catches degenerate/failed bins like the 9100-9200
    # dip to ~2).
    with np.errstate(all="ignore"):
        logb = np.log(np.clip(box3, 1e-3, None))
    fin = np.isfinite(logb)
    trend = np.polyval(np.polyfit(lam[fin], logb[fin], 2), lam) if fin.sum() >= 3 else np.full_like(logb, np.nanmedian(logb))
    resid = logb - trend
    rmad = np.nanmedian(np.abs(resid[fin] - np.nanmedian(resid[fin]))) or 1.0
    keep = fin & (np.abs(resid - np.nanmedian(resid[fin])) <= 4.0 * 1.4826 * rmad)
    kept = [r for r, k in zip(ok, keep) if k]
    n_rejected = int(np.sum(~keep))
    lam_k = np.array([r["lambda_A"] for r in kept])
    order = np.argsort(lam_k)
    param_table = {"lambda_A": [float(lam_k[i]) for i in order]}
    for name in PSFAO_PARAM_NAMES:
        vals = np.array([kept[i][name] for i in order])
        param_table[name] = [float(v) for v in vals]

    rings = np.array([r["ring_residual_pct"] for r in ok])
    # normalization round-trip on a fine model at one lambda
    norm_err = _norm_roundtrip(system, float(np.median(lam)), poly, norm_radius)

    psf_model = {"form": "psfao", "system": system_name, "smoothed_poly": poly,
                 "param_table": param_table, "n_bins_rejected": n_rejected,
                 "norm_radius_px": norm_radius, "fit_radius_px": fit_radius,
                 "param_names": list(PSFAO_PARAM_NAMES), "hybrid": False}
    if wave_bin_A is not None:
        psf_model["psfao_wave_bin_A"] = float(wave_bin_A)
        psf_model["psfao_wave_bin_A_note"] = (
            "Rejilla a la que _evaluate_psfao redondea lambda. Es el ancho de los bins "
            "que C1 ajusta: los parametros del PSD solo existen ahi, y una rejilla mas "
            "fina interpola entre dos bins parametros degenerados.")
    meta = {"ok": ok, "lam": lam, "rings": rings, "n_ok": len(ok),
            "n_rejected": n_rejected, "norm_err": norm_err, "poly": poly}
    return psf_model, meta


def _good_wave_mask(wavelengths, bad_windows_A):
    wave = np.asarray(wavelengths, dtype=np.float64)
    good = np.isfinite(wave)
    for lo, hi in bad_windows_A or ():
        good &= ~((wave >= float(lo)) & (wave <= float(hi)))
    return good


def make_psf_bins(wavelengths, *, bin_A=100.0, bad_windows_A=(), min_channels=3):
    wave = np.asarray(wavelengths, dtype=np.float64)
    good = _good_wave_mask(wave, bad_windows_A)
    if np.count_nonzero(good) < int(min_channels):
        raise RuntimeError("Too few good wavelength channels for PSF bins.")
    wmin = float(np.nanmin(wave[good]))
    wmax = float(np.nanmax(wave[good]))
    edges = np.arange(wmin, wmax + float(bin_A), float(bin_A))
    bins = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = good & (wave >= lo) & (wave < hi)
        if np.count_nonzero(mask) >= int(min_channels):
            bins.append(
                {
                    "wave_min_A": float(lo),
                    "wave_max_A": float(hi),
                    "wave_center_A": float(np.nanmedian(wave[mask])),
                    "indices": np.where(mask)[0],
                }
            )
    if not bins:
        raise RuntimeError("No valid PSF bins after wavelength masking.")
    return bins


def _median_image(cubes, indices):
    with np.errstate(all="ignore"):
        return np.nanmedian(cubes[:, indices, :, :], axis=(0, 1)).astype(np.float64)


def _positions_from_qc(qc):
    primary = tuple(map(float, qc["primary"]["pos_yx"]))
    companion = tuple(map(float, qc["companion"]["pos_yx"]))
    field = None
    if qc.get("field_source") is not None:
        field = tuple(map(float, qc["field_source"]["pos_yx"]))
    return primary, companion, field


def _fit_source_mask(shape, primary_yx, companion_yx, field_yx, mask_radius_px):
    centers = [companion_yx]
    if field_yx is not None:
        centers.append(field_yx)
    return source_mask(shape, centers, float(mask_radius_px))


def _detect_core_mask(image, primary_yx, cfg):
    threshold = cfg.get("psf_saturation_threshold")
    if threshold is None:
        return 0.0, False
    y, x = map(int, map(round, primary_yx))
    peak = float(image[y, x])
    if peak > float(threshold):
        return float(cfg.get("psf_core_mask_px", 2.0)), True
    return 0.0, False


def _row_from_fit(bin_index, bin_info, fit, metric, metric_hybrid=None):
    p = fit.params
    row = {
        "bin_index": int(bin_index),
        "wave_min_A": float(bin_info["wave_min_A"]),
        "wave_max_A": float(bin_info["wave_max_A"]),
        "wave_center_A": float(bin_info["wave_center_A"]),
        "n_channels": int(len(bin_info["indices"])),
        "success": bool(fit.success),
        "background": float(fit.background),
        "amplitude": float(p["amplitude"]),
        "y0": float(p["y0"]),
        "x0": float(p["x0"]),
        "fwhm_maj": float(p["fwhm_maj"]),
        "fwhm_min": float(p["fwhm_min"]),
        "theta_deg": float(p["theta_deg"]),
        "beta": float(p["beta"]),
        "chi2r": float(fit.chi2r),
        "clip_frac": float(fit.clip_frac),
        "n_fit": int(fit.n_fit),
        "ring_residual_pct": float(metric["median_pct"]),
        "ring_residual_p90_pct": float(metric["p90_pct"]),
    }
    for key, value in fit.errors.items():
        row[f"{key}_err"] = None if not np.isfinite(value) else float(value)
    if metric_hybrid is not None:
        row["ring_residual_pct_after_hybrid"] = float(metric_hybrid["median_pct"])
        row["ring_residual_p90_pct_after_hybrid"] = float(metric_hybrid["p90_pct"])
    return row


def _moffat_fit_rows(cubes, wavelengths, bins, positions_qc, cfg):
    primary_yx, companion_yx, field_yx = _positions_from_qc(positions_qc)
    fwhm_prelim = float(positions_qc.get("psf", {}).get("fwhm_px", cfg.get("psf_prelim_fwhm_px", 4.0)))
    mask_radius = float(cfg.get("psf_companion_mask_radius_px", cfg.get("psf_mask_radius_factor", 3.0) * fwhm_prelim))
    rows = []
    images = []
    models = []
    masks = []
    core_masks = []
    for i, bin_info in enumerate(bins):
        image = _median_image(cubes, bin_info["indices"])
        mask = _fit_source_mask(image.shape, primary_yx, companion_yx, field_yx, mask_radius)
        core_mask_px, saturation = _detect_core_mask(image, primary_yx, cfg)
        background = corner_background(image)
        fit = fit_moffat_image(
            image,
            center_yx=primary_yx,
            fit_radius_px=float(cfg.get("psf_fit_radius_px", 28.0)),
            mask=mask,
            background=background,
            core_mask_px=core_mask_px,
            sigma_clip=cfg.get("psf_sigma_clip", 3.0),
            max_iter=int(cfg.get("psf_max_clip_iter", 3)),
        )
        model = evaluate_moffat_fit(image.shape, fit)
        metric = companion_ring_metric(
            image,
            model,
            primary_yx,
            companion_yx,
            width_px=float(cfg.get("psf_companion_ring_width_px", 3.0)),
            source_exclusion_radius_px=mask_radius,
        )
        row = _row_from_fit(i, bin_info, fit, metric)
        row["core_mask_px"] = float(core_mask_px)
        row["saturation_detected"] = bool(saturation)
        rows.append(row)
        images.append(image)
        models.append(model)
        masks.append(mask)
        core_masks.append(core_mask_px)
    return rows, images, models, masks, {
        "mask_radius_px": mask_radius,
        "core_mask_px_max": float(np.nanmax(core_masks)) if core_masks else 0.0,
        "saturation_detected": bool(any(row["saturation_detected"] for row in rows)),
    }


def _apply_hybrid(ring_pcts, images, models, masks, primary_yx, companion_yx, fwhm_med, cfg):
    """Add the azimuthal-median residual (AO ring) hybrid term to the chosen
    form's per-bin models when the ring metric fails on >20% of bins (spec §3.5).

    Form-agnostic: ``models`` are per-bin model images (Moffat evaluations or
    Psfao reconstructions). The smoothing scale (>=2*FWHM) and the azimuthal
    symmetry of ``radial_hybrid_profile`` guarantee it cannot absorb the
    (masked) companion. Returns
    ``(models_hybrid, profiles, radii_ref, applied, after_pcts)``."""

    threshold = float(cfg.get("psf_hybrid_threshold_pct", 5.0))
    frac_limit = float(cfg.get("psf_hybrid_bin_fraction", 0.2))
    values = np.asarray(ring_pcts, dtype=np.float64)
    fail_frac = float(np.count_nonzero(values > threshold) / max(values.size, 1))
    if fail_frac <= frac_limit:
        return list(models), None, None, False, list(map(float, ring_pcts)), None

    fwhm_med = float(fwhm_med)
    smooth = max(2.0 * fwhm_med, float(cfg.get("psf_hybrid_smoothing_scale_factor", 2.0)) * fwhm_med)
    width = float(cfg.get("psf_companion_ring_width_px", 3.0))
    excl = float(cfg.get("psf_companion_mask_radius_px", cfg.get("psf_mask_radius_factor", 3.0) * fwhm_med))
    profiles = []
    radii_ref = None
    models_hybrid = []
    after_pcts = []
    after_p90s = []
    for image, model, mask in zip(images, models, masks):
        radii, profile = radial_hybrid_profile(
            image - model,
            primary_yx,
            mask=mask,
            smoothing_scale_px=smooth,
        )
        hybrid = evaluate_radial_profile(image.shape, primary_yx, radii, profile)
        model_h = model + hybrid
        metric_h = companion_ring_metric(
            image,
            model_h,
            primary_yx,
            companion_yx,
            width_px=width,
            source_exclusion_radius_px=excl,
        )
        after_pcts.append(float(metric_h["median_pct"]))
        after_p90s.append(float(metric_h["p90_pct"]))
        if radii_ref is None:
            radii_ref = radii
        if radii.size != radii_ref.size:
            profile = np.interp(radii_ref, radii, profile)
        profiles.append(profile)
        models_hybrid.append(model_h)
    # Safety guard: the hybrid term must reduce the ring residual. When the base
    # form already models the AO halo (Psfao), the azimuthal-residual term is
    # degenerate and — scaled by the inflated Moffat FWHM — can make the metric
    # WORSE. In that case discard it rather than corrupt a good model.
    before_med = float(np.nanmedian(values))
    after_med = float(np.nanmedian(after_pcts))
    if not (np.isfinite(after_med) and after_med < before_med):
        return list(models), None, None, False, list(map(float, ring_pcts)), None
    return models_hybrid, np.asarray(profiles, dtype=np.float32), radii_ref.astype(np.float32), True, after_pcts, after_p90s


## 4 · Chequeo de deriva


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/stats.py:finite_values": "8aa861655f2b",
    "musepipe/stats.py:finite_percentile": "c967dac05ce3",
    "musepipe/stats.py:robust_sigma": "ef2aa72a72de",
    "musepipe/reduction/verify.py:VerificationError": "d9acb4457343",
    "musepipe/reduction/verify.py:circular_aperture_mask": "b08d990cd3a2",
    "musepipe/reduction/verify.py:extract_aperture_spectrum": "e46a9616dd50",
    "musepipe/reduction/telluric.py:TelluricError": "5347f3394587",
    "musepipe/reduction/telluric.py:wavelength_axis_from_header": "f463214b3a31",
    "musepipe/psf.py:MoffatFit": "a2fef4724272",
    "musepipe/psf.py:moffat_alpha_from_fwhm": "8706bfcfbc81",
    "musepipe/psf.py:moffat_elliptical_profile": "eb8198bdfcf6",
    "musepipe/psf.py:moffat_image": "06ee63ae8460",
    "musepipe/psf.py:fixed_radius_grid": "f78dc6842678",
    "musepipe/psf.py:moffat_norm": "70e93f0c9d58",
    "musepipe/psf.py:normalized_moffat_psf": "25a3c40a75af",
    "musepipe/psf.py:source_mask": "21b7a974898a",
    "musepipe/psf.py:corner_background": "d31c01937f6c",
    "musepipe/psf.py:_initial_fit_params": "442371d033da",
    "musepipe/psf.py:_pack_params": "38287617cd80",
    "musepipe/psf.py:fit_moffat_image": "b041f49f8595",
    "musepipe/psf.py:evaluate_moffat_fit": "57172ee50751",
    "musepipe/psf.py:companion_ring_metric": "5b37f4cb618a",
    "musepipe/psf.py:smooth_parameter": "a255fe5c283a",
    "musepipe/psf.py:eval_smoothed_parameter": "05d5d9150afb",
    "musepipe/psf.py:build_psf_model_document": "d4644b048580",
    "musepipe/psf.py:_psfao_image_cached": "48bc78ca330c",
    "musepipe/psf.py:_evaluate_psfao": "ee69e821158f",
    "musepipe/psf.py:evaluate_psf_model": "749557eeec60",
    "musepipe/psf.py:psf_roundtrip_error": "8926b53d9db5",
    "musepipe/psf.py:radial_hybrid_profile": "24c26c3330fd",
    "musepipe/psf.py:evaluate_radial_profile": "2c72e505513d",
    "musepipe/stages/stage_e01_psfao.py:_bad_windows": "5a64b57438d1",
    "musepipe/stages/stage_e01_psfao.py:make_bins": "26d30a7ccd07",
    "musepipe/stages/stage_e01_psfao.py:_ring_residual": "8392c32f6e2f",
    "musepipe/stages/stage_e01_psfao.py:_box3_apcorr": "1e9e097f2c99",
    "musepipe/stages/stage_e01_psfao.py:fit_bin": "d8a97794858c",
    "musepipe/stages/stage_e01_psfao.py:_psfao_param_errors": "7e50d9dd0b26",
    "musepipe/stages/stage_e01_psfao.py:_psfao_fit_status": "f5ec233a3153",
    "musepipe/stages/stage_e01_psfao.py:fit_psfao_bins": "023110fabf46",
    "musepipe/stages/stage_e01_psfao.py:_norm_roundtrip": "57ef9141e86a",
    "musepipe/stages/stage_e01_psfao.py:build_psfao_model_document": "d3881076cde1",
    "musepipe/stages/stage_e01_psf.py:_good_wave_mask": "6943af46111a",
    "musepipe/stages/stage_e01_psf.py:make_psf_bins": "3736ec210a5e",
    "musepipe/stages/stage_e01_psf.py:_median_image": "403f92e60ac1",
    "musepipe/stages/stage_e01_psf.py:_positions_from_qc": "40304c29cfee",
    "musepipe/stages/stage_e01_psf.py:_fit_source_mask": "0ea3fb1c300f",
    "musepipe/stages/stage_e01_psf.py:_detect_core_mask": "8c2fef9053f8",
    "musepipe/stages/stage_e01_psf.py:_row_from_fit": "f0f2665848a6",
    "musepipe/stages/stage_e01_psf.py:_moffat_fit_rows": "9bb5ca2f37b8",
    "musepipe/stages/stage_e01_psf.py:_apply_hybrid": "f404f9c7ce82",
    "musepipe/psf.py:MOFFAT_BETA_FLOOR": "fc34677ec806",
    "musepipe/psf.py:PSF_SHAPE_PARAMS": "bbf6c549da2c",
    "musepipe/psf.py:_PSFAO_PARAM_NAMES": "d27321df07fa",
    "musepipe/stages/stage_e01_psfao.py:PSFAO_PARAM_NAMES": "67ebadb07841",
    "musepipe/stages/stage_e01_psfao.py:DEFAULT_X0": "b3224894ce4a"
}

def _pieza(cuerpo, name):
    """El nodo que define `name`: def/class, o la asignación de una constante.

    Las constantes también se vigilan: viajan copiadas igual que las
    funciones, y hasta ahora nadie comprobaba que siguieran siendo las de
    `musepipe` — añadir una banda a un diccionario dejaba esta copia atrás
    sin que nada lo dijera.
    """
    for n in cuerpo:
        if isinstance(n, (_ast.FunctionDef, _ast.ClassDef)) and n.name == name:
            inicio = min([n.lineno] + [d.lineno for d in n.decorator_list])
            return inicio, n.end_lineno
        if isinstance(n, _ast.Assign) and any(
                isinstance(t, _ast.Name) and t.id == name for t in n.targets):
            return n.lineno, n.end_lineno
        if (isinstance(n, _ast.AnnAssign) and isinstance(n.target, _ast.Name)
                and n.target.id == name):
            return n.lineno, n.end_lineno
    return None

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        sitio = _pieza(_ast.parse(text).body, name)
        if sitio is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        inicio, fin = sitio
        src = ''.join(lines[inicio - 1:fin]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera: python scripts/build_debug_notebooks.py --target {TARGET} C1')
else:
    print(f'sin deriva: las {len(_SHAS)} piezas copiadas son las de musepipe')


## 5 · El binado, que no es uno sino dos

Las dos formas se comparan «a igualdad de condiciones» (spec §3.4), pero **no binan igual**: `make_psf_bins` (Moffat) descarta las ventanas malas **canal a canal** y luego exige un mínimo de canales; `make_bins` (psfao) descarta el bin entero si su **punto medio** cae dentro. Con la misma anchura eso da un número distinto de bins, y es la razón de que el QC diga `n_bins` para uno y `model_comparison.moffat.n_bins` para el otro.


In [ ]:
BAD_PSFAO = _bad_windows(E01)
BINS_PSFAO = make_bins(WAVE_ALL, BIN_A, BAD_PSFAO, MIN_CANALES_BIN)
BINS_MOFFAT = make_psf_bins(WAVE_ALL, bin_A=BIN_A,
                            bad_windows_A=BAD_WINDOWS_MOFFAT,
                            min_channels=MIN_CANALES_BIN)
LAM_PSFAO = np.array([b[2] for b in BINS_PSFAO])
LAM_MOFFAT = np.array([b['wave_center_A'] for b in BINS_MOFFAT])

print(f'ventanas malas: psfao {BAD_PSFAO} · moffat {BAD_WINDOWS_MOFFAT}')
print(f'bins: psfao {len(BINS_PSFAO)} · moffat {len(BINS_MOFFAT)}')
print(f'   el QC dice: binning.n_bins = {QC_C1["binning"]["n_bins"]}'
      f' · model_comparison.moffat.n_bins ='
      f' {QC_C1["model_comparison"]["moffat"]["n_bins"]}')
_solo_moffat = [w for w in LAM_MOFFAT if np.min(np.abs(LAM_PSFAO - w)) > BIN_A / 2]
if _solo_moffat:
    print(f'   bins que solo tiene Moffat: {[round(w, 1) for w in _solo_moffat]}'
          ' — caen en la ventana mala pero conservan canales buenos')
if FILAS_PSFAO:
    assert len(FILAS_PSFAO) == len(BINS_PSFAO), (
        'el binado psfao recalculado no coincide con el CSV: '
        f'{len(BINS_PSFAO)} vs {len(FILAS_PSFAO)}')
assert len(FILAS_MOFFAT) == len(BINS_MOFFAT), (
    'el binado moffat recalculado no coincide con el CSV: '
    f'{len(BINS_MOFFAT)} vs {len(FILAS_MOFFAT)}')
print('el binado recalculado reproduce las filas de los CSV')

# Y de paso: los nombres que la §2 tuvo que escribir a mano (es anterior a la
# copia) son los de `musepipe`, vigilados por el chequeo de deriva.
assert NOMBRES == list(PSFAO_PARAM_NAMES if ES_PSFAO else PSF_SHAPE_PARAMS), (
    f'los nombres de la §2 se han desviado: {NOMBRES}')
_p, _c, _f = _positions_from_qc(POS)
assert (_p, _c, _f) == (STAR_YX, COMP_YX, FIELD_YX), (
    'la lectura de posiciones de la §2 ya no es la de `_positions_from_qc`')
print('los nombres y las posiciones de la §2 coinciden con la copia')


## 6 · El ajuste por bin, reajustado de verdad

Se reajustan `1 de cada PASO_BINS` bins más unos cuantos de los fallidos, con `fit_psfao_bins` tal cual la usa la cadena. Cada bin es independiente, así que los reajustados deben salir **idénticos** a los del CSV.

Del cubo se leen **solo los canales de esos bins** (memmap): el stack son 425 MB por extensión y no hace falta traerlo entero para ajustar seis ventanas.


In [ ]:
BINS = BINS_PSFAO if ES_PSFAO else BINS_MOFFAT
_lam_bin = ((lambda b: b[2]) if ES_PSFAO else (lambda b: b['wave_center_A']))
_canales_bin = ((lambda b: np.where(b[3])[0]) if ES_PSFAO
                else (lambda b: np.asarray(b['indices'])))
_por_lambda = {round(float(r[CLAVE_LAMBDA]), 3): r for r in FILAS}
_fallidos = [i for i, b in enumerate(BINS)
             if str(_por_lambda.get(round(float(_lam_bin(b)), 3), {})
                    .get('status', 'ok')) != 'ok']
IDX_BINS = sorted(set(range(0, len(BINS), PASO_BINS))
                  | set(_fallidos[:INCLUIR_FALLIDOS]))
print(f'se reajustan {len(IDX_BINS)} bins de {len(BINS)} (forma {FORMA_ELEGIDA}):'
      f' {[round(float(_lam_bin(BINS[i])), 1) for i in IDX_BINS]}')

_sel_total = np.zeros(WAVE_ALL.size, dtype=bool)
for i in IDX_BINS:
    _sel_total[_canales_bin(BINS[i])] = True
CANALES_SUB = np.where(_sel_total)[0]
with fits.open(CUBE_PATH, memmap=True) as h:
    _c = h['CUBES'].data if 'CUBES' in h else h[1].data
    _s = h['STAT'].data if 'STAT' in h else None
    if _c.ndim == 4:
        _c = _c[0]
        _s = None if _s is None else _s[0]
    CUBE_SUB = np.asarray(_c[CANALES_SUB], dtype=float)
    STAT_SUB = (np.ones_like(CUBE_SUB) if _s is None
                else np.asarray(_s[CANALES_SUB], dtype=float))
_pos = {int(c): k for k, c in enumerate(CANALES_SUB)}
print(f'leídos {CANALES_SUB.size} canales de {WAVE_ALL.size}'
      f' ({CUBE_SUB.nbytes / 1e6:.0f} MB)')

# maoppy imprime una línea por iteración y avisa de `alpha < 2*df` en cada
# bin: cientos de líneas que tapan el resultado. Se recogen y se resumen.
import contextlib, io as _io, warnings as _w
_ruido = _io.StringIO()
with contextlib.redirect_stdout(_ruido), _w.catch_warnings():
    _w.simplefilter('ignore')
    if ES_PSFAO:
        BINS_SUB = []
        for i in IDX_BINS:
            a, b, mid, sel = BINS_PSFAO[i]
            sub = np.zeros(CANALES_SUB.size, dtype=bool)
            sub[[_pos[int(c)] for c in np.where(sel)[0]]] = True
            BINS_SUB.append((a, b, mid, sub))
        FILAS_MIAS, RECONS = fit_psfao_bins(
            CUBE_SUB, STAT_SUB, WAVE_ALL[CANALES_SUB], BINS_SUB,
            SYSTEM, COMP_YX, MASK_RADIUS_PX, FIT_RADIUS_PX, field_yx=FIELD_YX)
    else:
        # `_moffat_fit_rows` espera el cubo con su eje de exposiciones y bins
        # con `indices`: se le da el recortado, con los índices remapeados.
        BINS_SUB = [dict(BINS_MOFFAT[i],
                         indices=np.array([_pos[int(c)]
                                           for c in BINS_MOFFAT[i]['indices']]))
                    for i in IDX_BINS]
        FILAS_MIAS, _IMGS, _MODS, _MSKS, META_MASK = _moffat_fit_rows(
            CUBE_SUB[None, ...], WAVE_ALL[CANALES_SUB], BINS_SUB, POS, E01)
        RECONS = {float(f['wave_center_A']): (i, m)
                  for f, i, m in zip(FILAS_MIAS, _IMGS, _MODS)}
print(f'({len(_ruido.getvalue().splitlines())} líneas de traza recogidas)')

_peor, _n = 0.0, 0
print()
_cab = 'status' if ES_PSFAO else 'success / clip_frac'
print(f'{"λ [Å]":>9s}  {_cab:28s} {"nfev":>5s}'
      f'  máx |Δ| en los {len(NOMBRES)} parámetros')
for fila in FILAS_MIAS:
    ref = _por_lambda[round(float(fila[CLAVE_LAMBDA]), 3)]
    _conv = str(fila.get('status', 'ok')) == 'ok'
    d = (max(abs(float(fila[n]) - float(ref[n])) / max(abs(float(ref[n])), 1e-30)
             for n in NOMBRES) if _conv else 0.0)
    _peor = max(_peor, d); _n += 1
    if ES_PSFAO:
        _est = str(fila['status'])[:28]
        _nfev = int(fila['optimizer_nfev'])
        _mismo = '✓' if str(fila['status']) == str(ref['status']) else '✗ DISTINTO'
    else:
        _est = f'{bool(fila["success"])} / {float(fila["clip_frac"]):.3f}'
        _nfev = int(fila['n_fit'])
        _mismo = '✓' if bool(fila['success']) == bool(ref['success']) else '✗ DISTINTO'
    print(f'{float(fila[CLAVE_LAMBDA]):9.1f}  {_est:28s}'
          f' {_nfev:5d}  {d:.3e}  {_mismo}')
print()
print(f'peor discrepancia relativa en {_n} bins: {_peor:.3e}')


## 7 · Los bins que no convergen, y por qué

El QC los cuenta (`fit.n_fit_failed`) pero no dice qué les pasa. Todos son la misma cosa: **`psffit` devolvió el vector de arranque intacto**, con `nfev = 2`. El arranque es uno solo, `DEFAULT_X0`, compartido por los 43 bins — así que allí donde la PSF se aleja de él, el optimizador no arranca.

Esto no es un detalle de contabilidad: cada bin caído es un **agujero en `param_table`**, y los agujeros son la materia prima de los escalones de la §8.


In [ ]:
if not ES_PSFAO:
    _cf = np.array([float(r['clip_frac']) for r in FILAS_MOFFAT])
    _ex = [r for r in FILAS_MOFFAT if not bool(r['success'])]
    print('esta cadena eligió Moffat, que no tiene la noción de bin caído:')
    print(f'   `least_squares` no convergió en {len(_ex)} de {len(FILAS_MOFFAT)} bins')
    print(f'   recorte sigma máximo {100 * np.nanmax(_cf):.2f} %'
          f' (el QC dice clip_frac_max = {QC_C1["fit"]["clip_frac_max"]:.4f})')
    print(f'   χ²ᵣ mediana {np.nanmedian([float(r["chi2r"]) for r in FILAS_MOFFAT]):.3g}')
    print()
    print('   -> sin bins caídos no hay agujeros en el modelo, y el suavizado')
    print('      cubre todo el rango. Los escalones de la §8 son de la otra forma.')
_malos = [r for r in FILAS_PSFAO if str(r.get('status', 'ok')) != 'ok']
if not FILAS_PSFAO:
    print('\n(no hay CSV de psfao en este run: la rama no llegó a escribirse)')
print()
print(f'{len(_malos)} de {len(FILAS_PSFAO)} bins de psfao sin converger'
      f' (el QC dice fit.n_fit_failed = {QC_C1["fit"].get("n_fit_failed", 0)})')
print()
print(f'{"λ [Å]":>9s}  {"status":32s} {"nfev":>5s} {"cost":>12s}')
for r in _malos:
    print(f'{r["lambda_A"]:9.1f}  {str(r["status"])[:32]:32s}'
          f' {int(r.get("optimizer_nfev", 0)):5d} {float(r.get("optimizer_cost", np.nan)):12.4g}')

_lam_malos = np.array([r['lambda_A'] for r in _malos]) if _malos else np.array([])
if _lam_malos.size:
    _corte = np.where(np.diff(_lam_malos) > BIN_A + 1)[0]
    _rachas = np.split(_lam_malos, _corte + 1)
    _larga = max(_rachas, key=len)
    print()
    print(f'la racha más larga son {len(_larga)} bins seguidos:'
          f' {_larga.min():.0f}-{_larga.max():.0f} Å'
          f' ({(_larga.max() - _larga.min()) + BIN_A:.0f} Å sin un solo ajuste)')
_ok = [r for r in FILAS_PSFAO if str(r.get('status', 'ok')) == 'ok']
if _ok:
    print()
    print('el vector de arranque, y lo que sale donde SÍ converge:')
    print(f'{"param":>8s} {"DEFAULT_X0":>14s} {"mediana ok":>14s}'
          f' {"mín":>14s} {"máx":>14s}')
    for k, n in enumerate(PSFAO_PARAM_NAMES):
        v = np.array([float(r[n]) for r in _ok])
        print(f'{n:>8s} {DEFAULT_X0[k]:14.6g} {np.nanmedian(v):14.6g}'
              f' {np.nanmin(v):14.6g} {np.nanmax(v):14.6g}')


## 8 · De dónde salen los saltos escalonados: `_evaluate_psfao`

Esta es la sección del notebook. Todo lo que la cadena hace con el modelo pasa por una sola función, `_evaluate_psfao`, que recibe una λ y devuelve una imagen de PSF. Entre la tabla de bins y esa imagen hay **cuatro** transformaciones, y las cuatro fabrican mesetas:

| # | qué hace | línea del original |
|---|---|---|
| 1 | redondea λ a múltiplos de `psfao_wave_bin_A`, que C1 escribe en el documento igual al ancho de bin que ajustó (**50 Å** solo en documentos viejos, que no la declaran) | `w_eff = round(λ/wave_bin)*wave_bin` |
| 2 | **interpola linealmente** `param_table` — no evalúa `smoothed_poly` | `np.interp(w, lam, table[name])` |
| 3 | recorta λ al rango de la tabla | `np.clip(w_eff, lam.min(), lam.max())` |
| 4 | recorta los parámetros a los límites físicos de Psfao | dentro de `_psfao_image_cached` |

Los cuatro son de la **rama psfao**. Si el objeto salió `moffat`, el modelo por canal es el polinomio suavizado evaluado en la λ exacta —sin redondeo, sin tabla, sin huecos— y la celda lo dice y mide lo que sí aplica.

El (1) es una **escalera literal**: unos 40 canales consecutivos comparten exactamente la misma PSF y luego salta. Existe por velocidad —convierte ~3681 construcciones FFT en ~90— y el comentario del original la justifica diciendo que «la PSF varía <0.5 % en ~50 Å». La celda mide si eso es cierto.

El (2) es el que desmiente al QC: `smoothing.per_param_model` dice `polynomial_deg2` para los siete parámetros, pero mientras exista `param_table` esa parábola **no se evalúa nunca**.


In [ ]:
if not ES_PSFAO:
    # Rama Moffat: no hay tabla, no hay redondeo de λ, no hay huecos. El
    # modelo por canal ES el polinomio, elegido por AIC entre grados 0-2 con
    # λ centrada y escalada. Los cuatro mecanismos de arriba no aplican, y
    # decirlo es parte de la auditoría.
    WAVE_BIN_A = 0.0
    W_EFF = WAVE_ALL.copy()
    W_USADA = WAVE_ALL.copy()
    LAM_T = np.array([float(r['wave_center_A']) for r in FILAS_MOFFAT])
    TABLA = {n: np.array([float(r[n]) for r in FILAS_MOFFAT]) for n in NOMBRES}
    PAR_CH = {n: np.array([eval_smoothed_parameter(PSF_MODEL['coefficients'][n], w)
                           for w in WAVE_ALL]) for n in NOMBRES}
    PAR_POLI = {}
    _huecos = []
    HUECO_MAX = None
    print('forma Moffat: el modelo por canal es el polinomio suavizado, evaluado')
    print('en la λ exacta. Sin redondeo, sin tabla que interpolar, sin huecos.')
    print()
    print(f'{"param":>10s} {"grado (AIC)":>12s} {"salto máx entre canales":>24s}')
    for n in NOMBRES:
        v = PAR_CH[n]
        s = np.nanmax(np.abs(np.diff(v)) / np.maximum(np.abs(v[:-1]), 1e-300))
        print(f'{n:>10s} {PSF_MODEL["coefficients"][n]["degree"]:12d}'
              f' {100 * s:22.4f} %')
    print()
    print('los cuatro mecanismos de la tabla de arriba son de la rama psfao.')
    print('Aquí el problema, si lo hay, es otro: el grado 2 no puede seguir una')
    print('curvatura real, y los bins atípicos no se rechazan (`outlier_bins` va')
    print('a [] escrito a mano, contra lo que pide la §3.3 del spec).')
else:
    WAVE_BIN_A = float(PSF_MODEL.get('psfao_wave_bin_A', 50.0))
    LAM_T = np.asarray(PSF_MODEL['param_table']['lambda_A'], dtype=float)
    TABLA = {n: np.asarray(PSF_MODEL['param_table'][n], dtype=float) for n in NOMBRES}
    # (1) la λ que de verdad ve el modelo
    W_EFF = (np.round(WAVE_ALL / WAVE_BIN_A) * WAVE_BIN_A if WAVE_BIN_A > 0
             else WAVE_ALL.copy())
    # (3) recortada al rango de la tabla
    W_USADA = np.clip(W_EFF, LAM_T.min(), LAM_T.max())
    # (2) los siete parámetros, canal a canal
    PAR_CH = {n: np.interp(W_USADA, LAM_T, TABLA[n]) for n in NOMBRES}
    # ...y la parábola que el QC anuncia y nadie evalúa
    POLI = PSF_MODEL.get('smoothed_poly') or {}
    PAR_POLI = {n: np.polyval(np.asarray(POLI[n], dtype=float), W_EFF)
                for n in NOMBRES if POLI.get(n)}

print()
print(f'canales del cubo          : {WAVE_ALL.size}')
if ES_PSFAO:
    print(f'λ efectivas tras el redondeo de {WAVE_BIN_A:.0f} Å : {np.unique(W_EFF).size}')
    print(f'nodos de param_table      : {LAM_T.size}'
          f'  ({LAM_T.min():.0f}-{LAM_T.max():.0f} Å)')
    print(f'  -> {WAVE_ALL.size / max(np.unique(W_EFF).size, 1):.0f} canales'
          ' comparten PSF, de media')
else:
    print(f'λ efectivas              : {np.unique(W_EFF).size} (sin redondeo)')
    print(f'bins ajustados           : {LAM_T.size}'
          f'  ({LAM_T.min():.0f}-{LAM_T.max():.0f} Å)')

# el salto más grande entre canales CONTIGUOS, que es lo que la escalera fabrica
_r0 = PAR_CH[NOMBRES[0]]
_salto = np.abs(np.diff(_r0)) / np.maximum(np.abs(_r0[:-1]), 1e-30)
# NO llamar a esto `_i`: en IPython `_i` es el código de la celda anterior
# (una cadena) y el kernel lo REASIGNA antes de ejecutar cada celda, así que
# el valor se pierde al pasar de celda y la siguiente revienta con
# `IndexError`. Los tests no lo cazan porque ejecutan con `exec` plano, sin
# kernel. Lo mismo vale para `_`, `__`, `_ii`, `In` y `Out`.
I_PEOR_SALTO = int(np.nanargmax(_salto))
_j = I_PEOR_SALTO
print()
print(f'mayor salto de `{NOMBRES[0]}` entre dos canales contiguos:')
print(f'   λ={WAVE_ALL[_j]:.1f} Å -> w_eff {W_EFF[_j]:.0f}, {NOMBRES[0]} = {_r0[_j]:.10g}')
print(f'   λ={WAVE_ALL[_j+1]:.1f} Å -> w_eff {W_EFF[_j+1]:.0f}, {NOMBRES[0]} = {_r0[_j+1]:.10g}')
print(f'   salto = {100 * _salto[_j]:.2f} % en {WAVE_ALL[_j+1] - WAVE_ALL[_j]:.2f} Å'
      + ('  (el comentario del original dice «<0.5 % en ~50 Å»)' if ES_PSFAO else ''))
print()
print('los siete, por lo que salta la escalera y por lo que abarcan en la tabla:')
print(f'{"param":>8s} {"salto máx entre canales":>24s} {"rango en param_table":>34s}')
for n in NOMBRES:
    v = PAR_CH[n]
    s = np.nanmax(np.abs(np.diff(v)) / np.maximum(np.abs(v[:-1]), 1e-300))
    t = TABLA[n]
    print(f'{n:>8s} {100 * s:22.2f} %'
          f' {f"{np.nanmin(t):.3g} … {np.nanmax(t):.3g}":>34s}')
if ES_PSFAO:
    print()
    print('`C` no es un escalón: es una DEGENERACIÓN. Recorre decenas de órdenes de')
    print('magnitud entre bins vecinos, que es lo que avisa el comentario de la propia')
    print('cadena («los parámetros de la PSD son degenerados, suavizarlos por separado')
    print('y luego reconstruir corrompe la PSF»). Interpolarlo linealmente tampoco lo')
    print('arregla: la tabla hereda la degeneración del ajuste por bin.')

if ES_PSFAO:
    # (3) los extremos que quedan congelados
    _azul = int(np.count_nonzero(W_EFF < LAM_T.min()))
    _rojo = int(np.count_nonzero(W_EFF > LAM_T.max()))
    print()
    print('canales fuera del rango de la tabla, con la PSF EXACTAMENTE congelada:')
    print(f'   {_azul} al azul (λ < {LAM_T.min():.0f} Å) y {_rojo} al rojo'
          f' (λ > {LAM_T.max():.0f} Å), de {WAVE_ALL.size}')

    # los huecos de la tabla: donde la «interpolación» es una recta larga
    _d = np.diff(LAM_T)
    _huecos = [(LAM_T[i], LAM_T[i + 1], _d[i]) for i in np.where(_d > BIN_A + 1)[0]]
    print()
    print(f'huecos de param_table (más de un bin de {BIN_A:.0f} Å seguido):')
    for a, b, w in sorted(_huecos, key=lambda t: -t[2]):
        _n_ch = int(np.count_nonzero((WAVE_ALL > a) & (WAVE_ALL < b)))
        print(f'   {a:.0f} -> {b:.0f} Å  ({w:.0f} Å, {_n_ch} canales'
              ' interpolados en línea recta)')
    HUECO_MAX = max(_huecos, key=lambda t: t[2]) if _huecos else None


In [ ]:
_filas_fig = (len(NOMBRES) + 2) // 2   # +1 hueco para la leyenda
fig, axes = plt.subplots(_filas_fig, 2, figsize=(11, 1.6 * _filas_fig + 2),
                         sharex=True, squeeze=False)
for ax in axes.ravel()[len(NOMBRES):]:
    ax.axis('off')
for ax, n in zip(axes.ravel(), NOMBRES):
    if HUECO_MAX is not None:
        ax.axvspan(HUECO_MAX[0], HUECO_MAX[1], color='tab:red', alpha=0.10, lw=0)
    for a, b, _w in _huecos:
        ax.axvspan(a, b, color='0.85', alpha=0.5, lw=0, zorder=0)
    ax.plot(WAVE_ALL, PAR_CH[n], lw=0.9, color='tab:blue',
            label='lo que evalúa la cadena')
    if n in PAR_POLI:
        ax.plot(WAVE_ALL, PAR_POLI[n], lw=1.0, ls='--', color='tab:orange',
                label='smoothed_poly (NO se evalúa)')
    ax.plot(LAM_T, TABLA[n], 'o', ms=3, color='k', label='nodos de param_table')
    ax.set_ylabel(n)
    ax.tick_params(labelsize=8)
axes.ravel()[-1].legend(*axes.ravel()[0].get_legend_handles_labels(),
                        loc='center', fontsize=9, frameon=False)
for ax in axes[-1]:
    ax.set_xlabel('λ [Å]')
fig.suptitle(
    ('los siete parámetros de Psfao: los nodos, la recta entre nodos,\n'
     'y la parábola que el QC anuncia y la cadena no usa') if ES_PSFAO else
    ('los seis parámetros de Moffat: los bins ajustados\n'
     'y el polinomio suavizado que sí se evalúa'), fontsize=10)
fig.tight_layout(); plt.show()

if ES_PSFAO:
    # el zoom donde se ve la escalera: 300 Å centrados en el peor salto
    _lo, _hi = (WAVE_ALL[I_PEOR_SALTO] - 150, WAVE_ALL[I_PEOR_SALTO] + 150)
    _m = (WAVE_ALL >= _lo) & (WAVE_ALL <= _hi)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.step(WAVE_ALL[_m], PAR_CH[NOMBRES[0]][_m], where='mid', lw=1.2,
            color='tab:blue')
    ax.plot(WAVE_ALL[_m], PAR_CH[NOMBRES[0]][_m], '.', ms=2, color='tab:blue')
    for w in np.unique(W_EFF[_m]):
        ax.axvline(w, color='0.8', lw=0.6, zorder=0)
    ax.set_xlabel('λ [Å]'); ax.set_ylabel(NOMBRES[0])
    ax.set_title(f'la escalera de {WAVE_BIN_A:.0f} Å, de cerca'
                 ' (líneas grises: los w_eff)', fontsize=9)
    fig.tight_layout(); plt.show()


### 8.b · El cuarto recorte: los límites físicos de Psfao

`_psfao_image_cached` recorta los siete parámetros a los límites del modelo antes de construir la imagen, con un comentario que lo dice: «los parámetros suavizados o interpolados pueden salirse de rango en los bordes y en los huecos». Es un salvavidas razonable — y también un **quinto generador de mesetas**, porque un parámetro pegado a su límite deja de variar con λ.

Nadie lo ha contado hasta ahora. La celda lo cuenta.


In [ ]:
from maoppy.psfmodel import Psfao

if not ES_PSFAO:
    print('la forma elegida es Moffat: `_psfao_image_cached` no interviene y no')
    print('hay límites físicos que recorten. Esta sección es de la rama psfao.')
_npix_apcorr = 2 * (int(np.ceil(NORM_RADIUS_PX)) + 2)
_unicas = np.unique(W_USADA) if ES_PSFAO else np.array([])
_pegados = {n: 0 for n in NOMBRES}
_cuantos_canales = 0
for w in _unicas:
    _m = Psfao((_npix_apcorr, _npix_apcorr), system=SYSTEM,
               samp=float(muse_nfm.samp(float(w) * 1e-10)))
    _low, _high = _m.bounds
    _n_ch = int(np.count_nonzero(W_USADA == w))
    _alguno = False
    for k, n in enumerate(NOMBRES):
        v = float(np.interp(w, LAM_T, TABLA[n]))
        fuera = ((np.isfinite(_low[k]) and v <= _low[k] + 1e-6)
                 or (np.isfinite(_high[k]) and v >= _high[k] - 1e-6))
        if fuera:
            _pegados[n] += _n_ch
            _alguno = True
    if _alguno:
        _cuantos_canales += _n_ch
print(f'evaluados los {_unicas.size} w_eff distintos EFECTIVOS en la rejilla de'
      f' apcorr ({_npix_apcorr}×{_npix_apcorr} px)')
print(f'   (de los {np.unique(W_EFF).size} que salen del redondeo; los de fuera del'
      ' rango de la tabla colapsan al recortar)')
print()
for n in NOMBRES:
    if _pegados[n]:
        print(f'   `{n}` queda pegado a su límite físico en {_pegados[n]} canales'
              f' ({100 * _pegados[n] / WAVE_ALL.size:.1f} %)')
if _cuantos_canales:
    print(f'\n   en total {_cuantos_canales} de {WAVE_ALL.size} canales'
          f' ({100 * _cuantos_canales / WAVE_ALL.size:.1f} %) llevan al menos un'
          ' parámetro recortado:\n   ahí la PSF deja de responder a λ.')
elif ES_PSFAO:
    print('   ningún parámetro toca sus límites: este recorte no fabrica mesetas'
          ' en este run.')


## 9 · La consecuencia observable: la curva de crecimiento

Los parámetros escalonados no se ven en un espectro; lo que se ve es su efecto en **cómo reparte el modelo la luz entre el núcleo y el halo**, que es exactamente lo que consume la cadena: C4 ajusta un coeficiente que *es* el flujo del modelo dentro de `norm_radius_px`, y C2 divide por una apcorr sacada del mismo modelo.

Como el modelo está normalizado a 1 dentro de `norm_radius_px`, la fracción encerrada `F(r)` **es** la curva de crecimiento, y `F(25) = 1` por construcción. Se calcula en los ~90 `w_eff` distintos y se reparte a los 3681 canales: dentro de un `w_eff` el modelo es idéntico, así que esto no es un submuestreo — es la misma información, sin repetir la FFT 40 veces.

Y de paso sale el **quinto** mecanismo, el que no está en la lista de la §8: `_evaluate_psfao` elige el tamaño de rejilla según los desplazamientos que le pidas. Con offsets dentro de `norm_radius` construye una rejilla pequeña; con offsets mayores, una de ~284 px. Son **dos discretizaciones distintas de la misma PSF**, y la celda mide en cuánto discrepan.


In [ ]:
def fraccion_encerrada(model_doc, waves, radios, radio_rejilla):
    """`F(r)` del modelo en cada λ, sobre una rejilla de ±radio_rejilla px."""
    R = int(np.ceil(radio_rejilla))
    dy, dx = np.mgrid[-R:R + 1, -R:R + 1].astype(float)
    rr = np.hypot(dy, dx)
    fuera = {r: rr <= float(r) for r in radios}
    out = {r: np.empty(len(waves)) for r in radios}
    for j, w in enumerate(waves):
        img = evaluate_psf_model(model_doc, float(w), dy, dx)
        for r in radios:
            out[r][j] = float(np.nansum(img[fuera[r]]))
    return out

# En psfao el modelo es constante dentro de cada `w_eff`, así que basta con
# evaluar los ~90 distintos y repartirlos: es la misma información, sin repetir
# la FFT 40 veces. En Moffat no hay redondeo, así que se submuestrea en λ.
if ES_PSFAO:
    _UNI, _INV = np.unique(W_USADA, return_inverse=True)
else:
    _paso = max(1, WAVE_ALL.size // 200)
    _UNI = WAVE_ALL[::_paso]
    _INV = np.clip(np.arange(WAVE_ALL.size) // _paso, 0, _UNI.size - 1)
_R_MAX = max(RADIOS_PX)
GC_GRANDE = fraccion_encerrada(PSF_MODEL, _UNI, RADIOS_PX, _R_MAX)
_RAD_PEQ = tuple(r for r in RADIOS_PX if r <= NORM_RADIUS_PX)
GC_PEQUE = fraccion_encerrada(PSF_MODEL, _UNI, _RAD_PEQ, NORM_RADIUS_PX)

print(f'rejilla grande: offsets hasta {_R_MAX:.0f} px > norm_radius'
      f' {NORM_RADIUS_PX:.0f} -> la del régimen psffit')
print(f'rejilla pequeña: offsets hasta {NORM_RADIUS_PX:.0f} px'
      ' -> la del régimen apcorr/curva de crecimiento')
print()
print(f'{"r [px]":>7s} {"F(r) mediana":>14s} {"variación en λ":>16s}'
      f' {"discrepancia entre rejillas":>28s}')
for r in RADIOS_PX:
    v = GC_GRANDE[r][_INV]
    _rango = f'{100 * (np.nanmax(v) / np.nanmin(v) - 1):.2f} %'
    if r in GC_PEQUE:
        _dis = np.abs(GC_PEQUE[r] / GC_GRANDE[r] - 1.0)
        _txt = f'{100 * np.nanmax(_dis):.2f} % (máx)'
    else:
        _txt = '—'
    print(f'{r:7.0f} {np.nanmedian(v):14.4f} {_rango:>16s} {_txt:>28s}')
print()
print(f'F({NORM_RADIUS_PX:.0f}) debe ser 1 por normalización:'
      f' máx |F-1| = {np.nanmax(np.abs(GC_GRANDE[NORM_RADIUS_PX] - 1)):.2e}'
      f' (el QC declara roundtrip_error ='
      f' {QC_C1["normalization"]["roundtrip_error"]:.2e})')


In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1]})
for a, b, _w in _huecos:
    for ax in (a1, a2):
        ax.axvspan(a, b, color='0.85', alpha=0.6, lw=0, zorder=0)
for r in RADIOS_PX:
    a1.plot(WAVE_ALL, GC_GRANDE[r][_INV], lw=0.9, label=f'r = {r:.0f} px')
a1.axhline(1.0, color='k', lw=0.6, ls=':')
a1.set_ylabel('F(r), fracción encerrada')
a1.legend(fontsize=8, ncol=3)
a1.set_title('la curva de crecimiento del modelo entregado, canal a canal\n'
             '(gris: los huecos de param_table)', fontsize=10)
_ref = GC_GRANDE[min(RADIOS_PX)][_INV]
a2.plot(WAVE_ALL, _ref / np.nanmedian(_ref), lw=0.9, color='tab:red')
a2.set_ylabel(f'F({min(RADIOS_PX):.0f}) / mediana')
a2.set_xlabel('λ [Å]')
a2.set_title('el reparto núcleo/halo, normalizado: aquí se ven las mesetas',
             fontsize=9)
fig.tight_layout(); plt.show()


### 9.b · Contra el dato: el coeficiente del psffit ÷ la apertura real

La prueba limpia del traspaso del 2026-08-08, aquí en código y no en prosa. El coeficiente que ajusta C4 **es** el flujo del modelo dentro de `norm_radius_px`; la apertura al mismo radio sobre el cubo es la misma cantidad medida. Si el reparto núcleo/halo del modelo fuese correcto en λ, el cociente sería plano.

Dos cuidados que el original no tenía:

* la apertura se hace sobre **el cubo que C4 ajustó** (`stage02_xcorr_cube_stack.fits`), no sobre el de entrada de B1;
* al producto se le **quita la apcorr** (`flux / apcorr`) para quedarse con el coeficiente desnudo: `spec_psffit_star.fits` ya lleva multiplicado el factor total/normrad de la curva de crecimiento, y compararlo sin quitarlo mezcla dos correcciones distintas.


In [ ]:
from musepipe.extraction.product import SpectrumProduct

EST = SpectrumProduct.read(SD / 'spec_psffit_star.fits')
_apc = (np.ones(EST.flux.size) if EST.apcorr is None
        else np.asarray(EST.apcorr, dtype=float))
COEF = np.asarray(EST.flux, dtype=float) / _apc

_R_VEN = int(np.ceil(max(RADIOS_PX))) + 5
_y0, _x0 = int(round(STAR_YX[0])), int(round(STAR_YX[1]))
with fits.open(CUBE_PATH, memmap=True) as h:
    _c = h['CUBES'].data if 'CUBES' in h else h[1].data
    if _c.ndim == 4:
        _c = _c[0]
    _ny, _nx = _c.shape[1], _c.shape[2]
    _sy = slice(max(0, _y0 - _R_VEN), min(_ny, _y0 + _R_VEN + 1))
    _sx = slice(max(0, _x0 - _R_VEN), min(_nx, _x0 + _R_VEN + 1))
    VENTANA = np.asarray(_c[:, _sy, _sx], dtype=float)
_yx_ven = (STAR_YX[0] - _sy.start, STAR_YX[1] - _sx.start)
print(f'ventana {VENTANA.shape} alrededor de la primaria'
      f' ({VENTANA.nbytes / 1e6:.0f} MB)')

APER = {r: extract_aperture_spectrum(VENTANA, _yx_ven, float(r)) for r in RADIOS_PX}
_fin = np.isfinite(COEF) & (COEF > 0)
print()
print(f'{"r [px]":>7s} {"coef/apertura":>15s} {"p2-p98":>16s}'
      f' {"lo dice el modelo":>18s} {"con apcorr dentro":>18s}')
for r in RADIOS_PX:
    c = COEF / APER[r]
    f = _fin & np.isfinite(c) & (c > 0)
    _p2, _p98 = np.nanpercentile(c[f], [2, 98])
    _mod = float(np.nanmedian(1.0 / GC_GRANDE[r][_INV]))
    _con = float(np.nanmedian((np.asarray(EST.flux, dtype=float) / APER[r])[f]))
    print(f'{r:7.0f} {np.nanmedian(c[f]):15.3f} {f"{_p2:.3f}-{_p98:.3f}":>16s}'
          f' {_mod:18.3f} {_con:18.3f}')
print()
print('la columna «lo dice el modelo» es 1/F(r): lo que el cociente valdría si el')
print('modelo repartiera la luz como el dato. La diferencia entre las dos columnas')
print('es el error del reparto núcleo/halo, y crece hacia el núcleo.')
print()
print('La última columna es el mismo cociente SIN quitar la apcorr, que es como se')
print('midió en la sesión del 2026-08-08 (docs/2026-08-08_handoff.md §3). Sirve para')
print('enlazar con aquellos números; el de la segunda columna es el limpio, porque')
print('la apcorr es otra corrección y además depende de λ.')

_c25 = COEF / APER[NORM_RADIUS_PX] if NORM_RADIUS_PX in APER else COEF / APER[RADIOS_PX[3]]
_f = _fin & np.isfinite(_c25) & (_c25 > 0)
fig, ax = plt.subplots(figsize=(9, 3.4))
for a, b, _w in _huecos:
    ax.axvspan(a, b, color='0.85', alpha=0.6, lw=0, zorder=0)
ax.plot(np.asarray(EST.wave_A)[_f], _c25[_f], lw=0.5, color='0.6')
_k = 51
if _f.sum() > _k:
    _w_s = np.asarray(EST.wave_A)[_f]
    _v = _c25[_f]
    _suave = np.array([np.nanmedian(_v[max(0, j - _k // 2):j + _k // 2 + 1])
                       for j in range(_v.size)])
    ax.plot(_w_s, _suave, lw=1.4, color='tab:blue', label=f'mediana móvil {_k} ch')
    ax.legend(fontsize=8)
ax.set_xlabel('λ [Å]')
ax.set_ylabel(f'coef psffit / apertura r={NORM_RADIUS_PX:.0f} px')
ax.set_title('meseta, escalón, meseta: el reparto núcleo/halo del modelo, en λ',
             fontsize=10)
fig.tight_layout(); plt.show()


## 10 · Los parámetros de las dos formas, cubo a cubo

Hasta aquí todo mira **un** cubo: el combinado sobre el que ajusta la cadena. Esta sección abre la otra dimensión, la que nadie ha mirado nunca — **la exposición** — y lo hace para las **dos formas de PSF a la vez**.

> **Aquí se ajustan Moffat y Psfao siempre**, gane quien gane. Es a propósito y es lo contrario que el resto del notebook, que audita solo la forma entregada: la comparación entre las dos *es* el objetivo de esta sección.

**Por qué importa, y no es curiosidad.** La §8 dejó los escalones atribuidos a la degeneración de `C`/`alpha`, al recorte a los límites físicos y a la cuantización de λ a 50 Å. Lo que ninguna de esas medidas puede decidir es **cuánto de la estructura en λ es física y cuánto es ruido del ajuste**, porque con un solo cubo no hay con qué comparar. La dispersión entre exposiciones sí lo decide:

* si a λ fija los cubos se dispersan **tanto como** saltan los bins contiguos, los escalones son ruido y suavizar en λ está justificado;
* si la estructura en λ **se repite** en todas las exposiciones, es cromática y de verdad, y suavizarla sería borrar señal.

La §10.d lo mide y lo dice con número.

**Qué es «un cubo» aquí.** Los cubos **por exposición** que el config declara en `perexp_cubes` / `perexp_dir`, nunca por convención de nombres sobre un disco. Un objeto que no los declare se salta la sección entera; para tenerlos, `python scripts/regen_perexp_cubes.py --step all`.


In [ ]:
# El config del run, para los cubos por exposición (`_sibling_setting` cubre
# las cadenas repartidas entre runs, igual que en el notebook de D2).
CFG = json.loads((RD / 'config' / 'config.json').read_text(encoding='utf-8'))['config']
LEGACY = bool(json.loads((RD / 'config' / 'config.json')
                         .read_text(encoding='utf-8')).get('meta', {}).get('legacy'))

def _etiquetas(rutas):
    """Un nombre corto y UNICO por cubo, sacado de su directorio.

    Los arboles de reduccion no comparten convencion: unos usan `exp1..expN`
    y otros el identificador de la exposicion
    (`2022-08-29_MUSE.2022-08-29T23:22:03.446`). Tomar `p.stem` daria
    `DATACUBE_FINAL` para TODOS. Es la misma funcion que usa D2.
    """
    crudas = [p.parent.name for p in rutas]
    if all(c.lower().startswith('exp') for c in crudas) and len(set(crudas)) == len(crudas):
        return crudas
    import re as _re
    salida, cuenta = [], {}
    for c in crudas:
        m = _re.match(r'(\d{4})-(\d{2})-(\d{2})', c)
        noche = f'{m.group(2)}-{m.group(3)}' if m else c[:8]
        cuenta[noche] = cuenta.get(noche, 0) + 1
        salida.append(f'{noche}#{cuenta[noche]:02d}')
    return salida if len(set(salida)) == len(salida) else crudas

def _cubos_por_observacion():
    lista = CFG.get('perexp_cubes') or nb._sibling_setting(RUN_ID, 'perexp_cubes')
    if not lista:
        raiz = CFG.get('perexp_dir') or nb._sibling_setting(RUN_ID, 'perexp_dir')
        if raiz:
            lista = sorted(str(p) for p in Path(raiz).glob('*/DATACUBE_FINAL.fits'))
    presentes = [Path(p) for p in (lista or []) if Path(p).exists()]
    return list(zip(_etiquetas(presentes), presentes))

def _cabecera_de(cubo):
    """Lo que la exposicion sabe de si misma. `None` donde no lo declara."""
    with fits.open(cubo, memmap=True) as h:
        pri = h[0].header
        hdu = h['DATA'] if 'DATA' in h else h[1]
        forma = tuple(int(v) for v in hdu.shape)
    _x = [pri.get('HIERARCH ESO TEL AIRM START'), pri.get('HIERARCH ESO TEL AIRM END')]
    _x = [float(v) for v in _x if v is not None]
    return {'date_obs': str(pri.get('DATE-OBS') or ''),
            'exptime': pri.get('EXPTIME'),
            'airmass': float(np.mean(_x)) if _x else None,
            'seeing': pri.get('HIERARCH ESO TEL AMBI FWHM START'),
            'ob_id': pri.get('HIERARCH ESO OBS ID'),
            # El angulo del rotador. Los cubos salen todos con el norte
            # arriba (el DRS los remuestrea), asi que POSANG no cambia la
            # geometria del recorte; sirve para interpretar `theta` en la
            # §10.c, que es lo unico que deberia seguirlo.
            'posang': pri.get('HIERARCH ESO ADA POSANG'),
            'forma': forma}

OBS_C1 = _cubos_por_observacion()
if MAX_CUBOS is not None:
    OBS_C1 = OBS_C1[:int(MAX_CUBOS)]
CABECERAS = {etq: _cabecera_de(p) for etq, p in OBS_C1}
# El orden del eje Y del mapa: por fecha de observación, no por nombre.
OBS_C1 = sorted(OBS_C1, key=lambda ep: CABECERAS[ep[0]]['date_obs'] or ep[0])

if not OBS_C1:
    print('Este objeto no declara cubos por exposición: la §10 no se puede hacer.')
    print('Para tenerlos, añade al config del run  "perexp_cubes": [...]  o')
    print('"perexp_dir": "..."  y genera los cubos con')
    print('   python scripts/regen_perexp_cubes.py --step all')
else:
    print(f'{len(OBS_C1)} cubos por exposición, ordenados por DATE-OBS:\n')
    _cab = ('obs', 'DATE-OBS', 't[s]', 'X', 'DIMM["]', 'POSANG', 'OB', 'forma')
    print('{:<10s}{:<21s}{:>7s}{:>7s}{:>9s}{:>9s}{:>10s}  {}'.format(*_cab))
    for _etq, _p in OBS_C1:
        _m = CABECERAS[_etq]
        print('{:<10s}{:<21s}{:>7s}{:>7s}{:>9s}{:>9s}{:>10s}  {}'.format(
            _etq, (_m['date_obs'] or 'n/d')[:19],
            'n/d' if _m['exptime'] is None else f"{float(_m['exptime']):.0f}",
            'n/d' if _m['airmass'] is None else f"{_m['airmass']:.3f}",
            'n/d' if _m['seeing'] is None else f"{float(_m['seeing']):.2f}",
            'n/d' if _m['posang'] is None else f"{float(_m['posang']):.2f}",
            str(_m['ob_id'] or 'n/d'), _m['forma']))
    _obs = sorted({str(CABECERAS[e]['ob_id']) for e, _ in OBS_C1})
    print(f'\nOB distintos: {len(_obs)} ({", ".join(_obs)})'
          '  — dos OB son dos noches, y promediarlos es una decisión')


### 10.a · El ajuste, y las tres cosas que hay que resolver

Un cubo por exposición **no es** el stack de B2: es un `DATACUBE_FINAL` en su propio marco de píxeles, más grande y con la estrella donde le toque. Tres ajustes de encuadre, ninguno de física:

1. **`fit_bin` supone la primaria en el centro del array** (`cy, cx = ny//2, nx//2`). En el cubo de la cadena eso se cumple porque B1 recorta con la primaria en el centro. Así que de cada exposición se recorta una ventana **del mismo tamaño** centrada en su propia primaria, localizada por máximo + centroide de la imagen blanca — la misma receta que usa D2. El centrado es a píxel entero (≤ 0.5 px de resto), y ese resto lo absorbe el `dxdy` del propio ajuste, igual que en la cadena.
2. **Dónde cae el compañero.** Se toma su desplazamiento respecto del **centro del array** en el cubo de la cadena —que es lo que `fit_bin` usa de verdad para enmascarar y para medir el anillo— y se aplica igual en cada ventana. Vale porque el DRS remuestrea todas las exposiciones a la misma orientación: `CD1_2 = 0` en todas, aunque el rotador (`ADA POSANG`) se mueva entre ellas.
3. **Bins y radios** son los de la cadena, del config resuelto (`E01`), no literales. Los bins se recalculan sobre el eje λ **de cada cubo**.

Se lee solo lo necesario: los canales de los bins elegidos, ya recortados a la ventana. `STAT` está en todas las exposiciones, así que los pesos del ajuste son los mismos que usa la cadena.

> **La caché.** Esto cuesta ~4.5 s por ajuste de Psfao (0.3 s los de Moffat), así que el resultado se guarda en `runs/<RUN>/tables/c1_perexp_psf_params.json` y las siguientes ejecuciones lo leen en menos de un segundo. La clave de la caché incluye las rutas y fechas de los cubos, los bins, los radios, la posición del compañero y los `sha` de las funciones copiadas: si cambia cualquiera de esas cosas, se recalcula solo. Para forzarlo, `RECALCULAR_MAPA = True`; para tirarla, borra el fichero. **Es el único notebook `debug` que escribe en disco**, y no escribe nada si el run está marcado `legacy`.


In [ ]:
import contextlib as _ctx, io as _io, time as _time, warnings as _w

# Los siete de Psfao salen de la constante copiada, no del modelo en disco:
# aquí se ajustan LAS DOS formas gane quien gane, y en un objeto donde
# ganó Moffat el `psf_model.json` no trae `param_names` de psfao.
PARAMS_PSFAO = list(PSFAO_PARAM_NAMES)
PARAMS_MOFFAT = ['fwhm_maj', 'fwhm_min', 'theta_deg', 'beta', 'amplitude']
CACHE_MAPA = RD / 'tables' / 'c1_perexp_psf_params.json'

# El compañero, respecto del CENTRO DEL ARRAY: es lo que `fit_bin` usa.
_ny_cad, _nx_cad = int(FORMA_CUBO[-2]), int(FORMA_CUBO[-1])
DESPL_COMP = (COMP_YX[0] - _ny_cad // 2, COMP_YX[1] - _nx_cad // 2)
DESPL_CAMPO = (None if FIELD_YX is None else
               (FIELD_YX[0] - _ny_cad // 2, FIELD_YX[1] - _nx_cad // 2))

def _clave_cache():
    """Todo lo que invalidaria el barrido. Si cambia, se recalcula."""
    return {
        'cubos': [[e, str(p), int(p.stat().st_mtime)] for e, p in OBS_C1],
        'bin_A': float(BIN_A_MAPA), 'paso_bins': int(PASO_BINS_MAPA),
        'min_canales': int(MIN_CANALES_BIN),
        'ventana': [_ny_cad, _nx_cad],
        'radios': [float(FIT_RADIUS_PX), float(MASK_RADIUS_PX)],
        'despl_comp': [round(v, 6) for v in DESPL_COMP],
        'despl_campo': (None if DESPL_CAMPO is None
                        else [round(v, 6) for v in DESPL_CAMPO]),
        # Si la matemática copiada cambia, el barrido guardado ya no es el
        # que produciría este notebook.
        'shas': {k: v for k, v in _SHAS.items()
                 if 'stage_e01' in k or 'psf.py' in k},
    }

def _primaria_y_ventana(hdu, paso=37):
    """(y0, x0) de la esquina de la ventana y (cy, cx) de la primaria."""
    # Los bordes del cubo son NaN en todos los canales: `nanmedian` avisa una
    # vez por columna vacía y llena la salida de ruido. Es esperado, no un
    # problema, y el máximo se busca igual.
    with _w.catch_warnings():
        _w.simplefilter('ignore', RuntimeWarning)
        blanco = np.nanmedian(np.asarray(hdu.data[::paso], dtype=np.float64), axis=0)
    py, px = (int(v) for v in np.unravel_index(np.nanargmax(blanco), blanco.shape))
    sy = slice(max(0, py - 3), py + 4); sx = slice(max(0, px - 3), px + 4)
    caja = np.nan_to_num(blanco[sy, sx])
    yy, xx = np.mgrid[sy.start:sy.start + caja.shape[0],
                      sx.start:sx.start + caja.shape[1]]
    cy = float((yy * caja).sum() / caja.sum())
    cx = float((xx * caja).sum() / caja.sum())
    ny, nx = hdu.shape[1:]
    # La ventana deja la primaria en (ny_cad//2, nx_cad//2), que es donde
    # `fit_bin` da por hecho que está.
    y0 = int(np.clip(round(cy) - _ny_cad // 2, 0, max(0, ny - _ny_cad)))
    x0 = int(np.clip(round(cx) - _nx_cad // 2, 0, max(0, nx - _nx_cad)))
    return (y0, x0), (cy, cx)

def _ajusta_un_cubo(etq, ruta):
    """Las filas de las DOS formas para un cubo por exposicion."""
    with fits.open(ruta, memmap=True) as h:
        hdu = h['DATA'] if 'DATA' in h else h[1]
        hst = h['STAT'] if 'STAT' in h else None
        (y0, x0), (cy, cx) = _primaria_y_ventana(hdu)
        wave = wavelength_axis_from_header(hdu.header, int(hdu.shape[0]))
        bins_p = make_bins(wave, BIN_A_MAPA, BAD_PSFAO, MIN_CANALES_BIN)
        bins_m = make_psf_bins(wave, bin_A=BIN_A_MAPA,
                               min_channels=MIN_CANALES_BIN,
                               bad_windows_A=BAD_WINDOWS_MOFFAT)
        idx_p = list(range(0, len(bins_p), PASO_BINS_MAPA))
        canales = np.zeros(wave.size, dtype=bool)
        for i in idx_p:
            canales[np.where(bins_p[i][3])[0]] = True
        # Los bins de Moffat no son los de psfao (§5): se eligen los que
        # caen dentro de los canales ya leídos, para no leer el cubo dos veces.
        idx_m = [j for j, b in enumerate(bins_m)
                 if canales[np.asarray(b['indices'])].all()]
        sel = np.where(canales)[0]
        ys, xs = slice(y0, y0 + _ny_cad), slice(x0, x0 + _nx_cad)
        # Por TRAMOS contiguos. Los bins elegidos van de un extremo del
        # espectro al otro, así que un solo `sel.min():sel.max()` se trae el
        # cubo entero —850 MB por extensión— para quedarse con la cuarta
        # parte. Así se lee solo lo que se ajusta.
        tramos = [(int(t[0]), int(t[-1]) + 1)
                  for t in np.split(sel, np.where(np.diff(sel) > 1)[0] + 1)]
        cubo = np.concatenate([np.asarray(hdu.data[i:j, ys, xs], dtype=float)
                               for i, j in tramos])
        stat = (np.ones_like(cubo) if hst is None else
                np.concatenate([np.asarray(hst.data[i:j, ys, xs], dtype=float)
                                for i, j in tramos]))
    pos = {int(c): k for k, c in enumerate(sel)}
    comp = (_ny_cad // 2 + DESPL_COMP[0], _nx_cad // 2 + DESPL_COMP[1])
    campo = (None if DESPL_CAMPO is None else
             (_ny_cad // 2 + DESPL_CAMPO[0], _nx_cad // 2 + DESPL_CAMPO[1]))
    filas = []
    with _ctx.redirect_stdout(_io.StringIO()), _w.catch_warnings():
        _w.simplefilter('ignore')
        subs = []
        for i in idx_p:
            aa, bb, mid, s = bins_p[i]
            m = np.zeros(sel.size, dtype=bool)
            m[[pos[int(c)] for c in np.where(s)[0]]] = True
            subs.append((aa, bb, mid, m))
        fp, _ = fit_psfao_bins(cubo, stat, wave[sel], subs, SYSTEM, comp,
                               MASK_RADIUS_PX, FIT_RADIUS_PX, field_yx=campo)
        for r in fp:
            filas.append({'cubo': etq, 'forma': 'psfao',
                          'lambda_A': float(r['lambda_A']),
                          'status': str(r.get('status', 'ok')),
                          'ring_residual_pct': float(r.get('ring_residual_pct', np.nan)),
                          **{k: float(r[k]) for k in PARAMS_PSFAO if k in r},
                          **{f'{k}_err': float(r.get(f'{k}_err', np.nan))
                             for k in PARAMS_PSFAO}})
        # Moffat necesita las posiciones EN LA VENTANA, con la forma del QC de B3.
        pos_qc = {'primary': {'pos_yx': [_ny_cad // 2, _nx_cad // 2]},
                  'companion': {'pos_yx': list(comp)},
                  'psf': {'fwhm_px': FWHM_PRELIM}}
        if campo is not None:
            pos_qc['field_source'] = {'pos_yx': list(campo)}
        bm = [dict(bins_m[j], indices=np.array([pos[int(c)]
                                               for c in bins_m[j]['indices']]))
              for j in idx_m]
        if bm:
            fm, _i, _mo, _ms, _me = _moffat_fit_rows(cubo[None, ...], wave[sel],
                                                     bm, pos_qc, E01)
            for r in fm:
                filas.append({'cubo': etq, 'forma': 'moffat',
                              'lambda_A': float(r['wave_center_A']),
                              'status': 'ok' if bool(r['success']) else 'fit_failed',
                              'ring_residual_pct': float(r.get('ring_residual_pct', np.nan)),
                              **{k: float(r[k]) for k in PARAMS_MOFFAT if k in r},
                              **{f'{k}_err': float(r.get(f'{k}_err', np.nan))
                                 for k in PARAMS_MOFFAT}})
    return filas, {'yx': [cy, cx], 'ventana': [y0, x0],
                   'n_bins_psfao': len(idx_p), 'n_bins_moffat': len(idx_m)}

FILAS_MAPA, GEOM_MAPA = [], {}
if OBS_C1:
    _clave = _clave_cache()
    # La caché guarda VARIAS rejillas, una por clave. Con una sola, alternar
    # `PASO_BINS_MAPA` entre 1 y 4 tiraba a la basura la corrida anterior:
    # 40 min de ida y 96 de vuelta. Se conservan las MAS_CACHES últimas.
    MAS_CACHES = 4
    _sha_clave = _hashlib.sha256(
        json.dumps(_clave, sort_keys=True).encode('utf-8')).hexdigest()[:12]
    _disco = {}
    if CACHE_MAPA.exists():
        try:
            _payload = json.loads(CACHE_MAPA.read_text(encoding='utf-8'))
        except ValueError:
            _payload = {}
        if isinstance(_payload, dict) and 'entradas' in _payload:
            _disco = _payload['entradas']
        elif isinstance(_payload, dict) and 'cache_key' in _payload:
            # Caché del formato viejo (una sola rejilla). Se convierte en vez
            # de tirarla: son horas de ajustes y la clave dice exactamente a
            # qué corresponde.
            _vieja = _hashlib.sha256(json.dumps(_payload['cache_key'],
                                                sort_keys=True).encode('utf-8')
                                     ).hexdigest()[:12]
            # El `ts` sale del fichero, NO es 0: con 0 la rejilla convertida
            # sería siempre la primera en ser desalojada, que es justo la que
            # más caro costó (es la que ya estaba).
            _disco = {_vieja: {'clave': _payload['cache_key'],
                               'geom': _payload.get('geom', {}),
                               'filas': _payload['filas'],
                               'ts': float(CACHE_MAPA.stat().st_mtime)}}
            print(f'caché en formato antiguo convertida ({_vieja},'
                  f' {len(_payload["filas"])} filas)')
    _guardado = None if RECALCULAR_MAPA else _disco.get(_sha_clave)
    if _guardado is None and _disco and not RECALCULAR_MAPA:
        print(f'la caché tiene {len(_disco)} rejilla(s) pero ninguna con esta clave'
              ' (cubos, bins, radios o')
        print('la matemática copiada han cambiado): se calcula y se AÑADE, sin'
              ' borrar las otras.')
    if _guardado is not None and not _guardado.get('completa', True):
        # Rejilla A MEDIAS: la corrida anterior se cortó (apagón, `kill`)
        # con parte de los cubos ya ajustados. Se reanuda desde ahí en vez
        # de repetirlos — cada cubo del barrido es independiente.
        FILAS_MAPA = list(_guardado['filas'])
        GEOM_MAPA = dict(_guardado.get('geom', {}))
        print(f'caché PARCIAL ({_sha_clave}): {len(GEOM_MAPA)} de'
              f' {len(OBS_C1)} cubos ya ajustados ({len(FILAS_MAPA)} filas).'
              ' Se reanuda por donde iba.')
        _guardado = None
    if _guardado is not None:
        FILAS_MAPA = _guardado['filas']
        GEOM_MAPA = _guardado.get('geom', {})
        print(f'caché leída ({_sha_clave}): {len(FILAS_MAPA)} filas'
              f' de {CACHE_MAPA}')
    else:
        _pendientes = [(e, p) for e, p in OBS_C1 if e not in GEOM_MAPA]
        _nb = len(range(0, len(BINS_PSFAO), PASO_BINS_MAPA))
        print(f'{len(_pendientes)} cubos × ~{_nb} bins × 2 formas'
              f' = ~{_nb * len(_pendientes)} ajustes Psfao a ~4.5 s, más los'
              ' de Moffat y la lectura. Esto se paga UNA vez: va a la caché.')

        def _guarda_cache(completa):
            """Escribe la rejilla, TERMINADA O A MEDIAS.

            Se llama después de CADA CUBO. Escribiendo solo al final, un
            barrido cortado valía lo mismo que no haberlo empezado: el
            2026-08-10 un apagado programado se llevó 2 h 39 min de ajustes
            al 90 %. Cada escritura es ~1 MB y medio segundo, 29 veces;
            frente a eso, cualquier corte cuesta como mucho UN cubo.
            """
            if LEGACY:
                return
            CACHE_MAPA.parent.mkdir(parents=True, exist_ok=True)
            _disco[_sha_clave] = {'clave': _clave, 'geom': GEOM_MAPA,
                                  'filas': FILAS_MAPA, 'ts': _time.time(),
                                  'completa': bool(completa)}
            while len(_disco) > MAS_CACHES:   # se va la más vieja; nunca
                _viejo = min(_disco,          # esta, que acaba de tocarse
                             key=lambda k: _disco[k].get('ts', 0))
                del _disco[_viejo]
            # Escritura ATÓMICA. Con 29 escrituras en vez de una, un corte
            # justo durante el `write_text` dejaría un JSON truncado, y eso
            # es peor que no tener caché: se pierden TODAS las rejillas.
            _tmp = CACHE_MAPA.with_name(CACHE_MAPA.name + '.tmp')
            _tmp.write_text(json.dumps({'version': 2, 'entradas': _disco},
                                       ensure_ascii=False), encoding='utf-8')
            _tmp.replace(CACHE_MAPA)

        if LEGACY:
            print('run marcado `legacy`: NO se escribe la caché, ni parcial.')
        _t0 = _time.time()
        for _k, (_etq, _p) in enumerate(_pendientes, 1):
            _tc = _time.time()
            _f, _g = _ajusta_un_cubo(_etq, _p)
            FILAS_MAPA.extend(_f); GEOM_MAPA[_etq] = _g
            _guarda_cache(False)   # <- el punto de control
            _malos = sum(1 for r in _f if r['status'] != 'ok')
            _t = _time.time() - _t0
            print(f'  [{_k:2d}/{len(_pendientes)}] {_etq:10s} {len(_f):3d}'
                  f' filas, {_malos:2d} sin converger,'
                  f' {_time.time() - _tc:5.1f} s  (total {_t / 60:.1f} min)')
            if _k == 1 and len(_pendientes) > 1:
                # El ETA de arriba solo cuenta los ajustes de Psfao. Con un
                # cubo medido ya se puede dar el de verdad.
                print(f'      -> a este ritmo, {_t * len(_pendientes) / 60:.0f}'
                      f' min para los {len(_pendientes)}')
        _guarda_cache(True)
        if not LEGACY:
            print(f'\ncaché escrita en {CACHE_MAPA}'
                  f' ({_sha_clave}; {len(_disco)} rejilla(s) guardadas)')

if FILAS_MAPA:
    # El eje λ va POR FORMA. Los dos binados no coinciden (§5: Moffat y
    # psfao no parten el espectro igual), y meterlos en una rejilla común
    # dejaba media matriz enmascarada en columnas alternas: el mapa salía
    # a rayas y la derivada en λ no medía nada.
    LAM_MAPA = {f: sorted({round(r['lambda_A'], 1) for r in FILAS_MAPA
                           if r['forma'] == f})
                for f in ('psfao', 'moffat')}
    ETQ_MAPA = [e for e, _ in OBS_C1 if any(r['cubo'] == e for r in FILAS_MAPA)]
    print(f'\nrejilla: {len(ETQ_MAPA)} cubos ×')
    for _f in ('psfao', 'moffat'):
        _r = [r for r in FILAS_MAPA if r['forma'] == _f]
        _ok = sum(1 for r in _r if r['status'] == 'ok')
        _l = LAM_MAPA[_f]
        _rango = f'{_l[0]:.0f}-{_l[-1]:.0f} Å' if _l else '—'
        print(f'   {_f:7s}: {len(_l):3d} columnas en λ ({_rango})'
              f' · {_ok}/{len(_r)} ajustes convergidos'
              f' ({100 * _ok / max(1, len(_r)):.0f} %)')
    # La primaria cae en un sitio distinto en cada exposición: eso es el
    # DITHER, es lo normal, y es justo por lo que la ventana se centra en
    # cada cubo por separado. Lo que sí rompería el ajuste es que la ventana
    # se saliera del cubo y hubiera que recortarla contra el borde: entonces
    # la primaria deja de estar en el centro del array y `fit_bin` —que da
    # por hecho que sí— mediría el anillo alrededor del sitio equivocado.
    _g = [(e, GEOM_MAPA[e]) for e in ETQ_MAPA if e in GEOM_MAPA]
    if _g:
        _yy = np.array([v['yx'] for _e, v in _g], dtype=float)
        _d = np.hypot(*(_yy - np.median(_yy, axis=0)).T)
        _recortadas = [e for e, v in _g
                       if v['ventana'] != [round(v['yx'][0]) - _ny_cad // 2,
                                           round(v['yx'][1]) - _nx_cad // 2]]
        print(f'\ndither: la primaria se mueve hasta {_d.max():.0f} px entre'
              f' exposiciones (mediana {np.median(_d):.0f} px).')
        print(f'ventanas de {_ny_cad}×{_nx_cad} px recortadas contra el borde:'
              f' {len(_recortadas)} de {len(_g)}')
        if _recortadas:
            print('   AVISO: en esas la primaria NO queda en el centro del array y'
                  ' el residuo')
            print(f'   de anillo no es comparable: {", ".join(_recortadas)}')
else:
    LAM_MAPA, ETQ_MAPA = {'psfao': [], 'moffat': []}, []


### 10.b · Estabilidad en λ, dos cubos

Cada parámetro contra λ, **con su barra de error**, para dos exposiciones. Por defecto la primera de cada noche (`CUBOS_LINEAS` lo cambia), porque el par interesante es el que cruza los dos OB. En gris, el ajuste del **cubo combinado** que hay en `stages/`, como referencia. Los ajustes que no convergen van con marcador hueco: un estancamiento tiene que verse, no desaparecer.

> **Las barras de las dos formas NO significan lo mismo.** Psfao propaga `1/sqrt(diag(JᵀJ))` con pesos `1/STAT`, y `docs/noise_model.md` tiene medido que STAT subestima el ruido: es una **cota inferior formal**. Moffat usa `pinv(JᵀJ)·σ²` con σ **empírica** del residuo. Sirven para juzgar la estabilidad *dentro* de cada forma; comparar el tamaño de una barra de Psfao con una de Moffat no significa nada.

`C` va en escala logarítmica porque recorre decenas de órdenes de magnitud entre bins vecinos (§8): no es una escala fea, es el síntoma.


In [ ]:
# Dos series + la referencia: colores fijos, asignados al cubo y no al orden
# de dibujo, y distinguibles también en visión con deficiencia de color.
COLOR_CUBO = ('#4C78A8', '#F58518')
COLOR_CADENA = '0.6'

def _serie(forma, cubo, campo):
    d = {round(r['lambda_A'], 1): r for r in FILAS_MAPA
         if r['forma'] == forma and r['cubo'] == cubo}
    lam = np.array(sorted(d))
    val = np.array([d[l].get(campo, np.nan) for l in lam], dtype=float)
    err = np.array([d[l].get(f'{campo}_err', np.nan) for l in lam], dtype=float)
    ok = np.array([d[l]['status'] == 'ok' for l in lam])
    return lam, val, err, ok

def _cadena(forma, campo):
    """La misma cantidad medida sobre el cubo combinado, si esta en disco."""
    filas = FILAS_PSFAO if forma == 'psfao' else FILAS_MOFFAT
    clave = 'lambda_A' if forma == 'psfao' else 'wave_center_A'
    buenas = [r for r in filas if str(r.get('status', 'ok')) == 'ok'
              and bool(r.get('success', True)) and campo in r]
    if not buenas:
        return np.array([]), np.array([])
    lam = np.array([float(r[clave]) for r in buenas])
    o = np.argsort(lam)
    return lam[o], np.array([float(buenas[i][campo]) for i in o])

def _dibuja_estabilidad(forma, campos, cubos):
    n = len(campos)
    fil = (n + 1) // 2
    fig, axes = plt.subplots(fil, 2, figsize=(11, 2.0 * fil), sharex=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, campo in zip(axes, campos):
        lc, vc = _cadena(forma, campo)
        if lc.size:
            ax.plot(lc, vc, lw=1.0, color=COLOR_CADENA, zorder=1,
                    label='cubo combinado (cadena)')
        for i, cubo in enumerate(cubos):
            lam, val, err, ok = _serie(forma, cubo, campo)
            if not lam.size:
                continue
            c = COLOR_CUBO[i % len(COLOR_CUBO)]
            e = np.where(np.isfinite(err), err, 0.0)
            ax.errorbar(lam[ok], val[ok], yerr=e[ok], color=c, lw=1.2, ms=4,
                        marker='o', capsize=2, elinewidth=0.8, label=cubo, zorder=3)
            if (~ok).any():
                ax.plot(lam[~ok], val[~ok], ls='none', marker='o', ms=5,
                        mfc='none', mec=c, mew=1.2, zorder=4)
        if campo == 'C' or campo == 'amplitude':
            ax.set_yscale('log')
        ax.set_ylabel(campo, fontsize=9)
        ax.grid(alpha=0.25, lw=0.5)
        ax.tick_params(labelsize=8)
    for ax in axes[n:]:
        ax.set_visible(False)
    for ax in axes[max(0, n - 2):n]:
        ax.set_xlabel('λ [Å]', fontsize=9)
    axes[0].legend(fontsize=7, ncol=3, loc='best')
    fig.suptitle(f'{forma}: los parámetros y su error, exposición a exposición',
                 fontsize=11)
    fig.tight_layout(); plt.show()

if FILAS_MAPA:
    if CUBOS_LINEAS:
        CUBOS_2 = [c for c in CUBOS_LINEAS if c in ETQ_MAPA]
    else:
        # El primero de cada noche: el par que cruza los dos OB.
        _vistas, CUBOS_2 = set(), []
        for _e in ETQ_MAPA:
            _n = (CABECERAS[_e]['date_obs'] or _e)[:10]
            if _n not in _vistas:
                _vistas.add(_n); CUBOS_2.append(_e)
        CUBOS_2 = CUBOS_2[:2] if len(CUBOS_2) >= 2 else ETQ_MAPA[:2]
    print('cubos de las curvas:', ', '.join(
        f"{c} ({(CABECERAS[c]['date_obs'] or '')[:19]})" for c in CUBOS_2))
    _dibuja_estabilidad('psfao', PARAMS_PSFAO, CUBOS_2)
    _dibuja_estabilidad('moffat', PARAMS_MOFFAT, CUBOS_2)

    # El residuo de anillo es lo ÚNICO comparable entre las dos
    # parametrizaciones, y es la figura de mérito con la que C1 elige.
    fig, ax = plt.subplots(figsize=(10, 3.2))
    for i, cubo in enumerate(CUBOS_2):
        for forma, ls in (('psfao', '-'), ('moffat', '--')):
            lam, val, _e, ok = _serie(forma, cubo, 'ring_residual_pct')
            if lam.size:
                ax.plot(lam[ok], val[ok], ls=ls, lw=1.3, marker='o', ms=3.5,
                        color=COLOR_CUBO[i % len(COLOR_CUBO)],
                        label=f'{cubo} · {forma}')
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('residuo de anillo [%]')
    ax.set_title('la figura de mérito con la que C1 elige forma '
                 '(continua psfao, discontinua moffat)', fontsize=10)
    ax.grid(alpha=0.25, lw=0.5); ax.legend(fontsize=7, ncol=2)
    fig.tight_layout(); plt.show()


### 10.c · El mapa: λ contra exposición

La vista global. Un panel por parámetro; **x = λ**, **y = el cubo ordenado por `DATE-OBS`**, color = el valor. La línea horizontal separa los OB.

El color es **divergente y centrado en la mediana** del propio parámetro, con los límites en percentiles robustos: la pregunta de esta sección no es «cuánto vale» sino «cuánto se aparta de lo típico», y un mapa divergente responde a eso directamente — el gris del centro *es* «como siempre». Las celdas de ajustes que no convergieron van en gris claro y **no** son un valor bajo: son un hueco.

Cómo se lee: **rayas verticales** (una columna entera desviada, en todas las exposiciones) = estructura cromática real. **Rayas horizontales** (una exposición entera desviada) = esa observación es distinta —seeing, masa de aire—. **Moteado sin estructura** = ruido de ajuste, y ahí suavizar en λ no borra nada.


In [ ]:
#: Los parámetros que son ORIENTACIONES, con su periodo. Una elipse vuelve
#: a ser la misma media vuelta después, así que 89° y −89° distan 2° y no
#: 178°. Sin envolver, `theta` salía con una dispersión del 200 % y un
#: veredicto de «estructura en λ real» que era puro artefacto de la resta.
PERIODO = {'theta': np.pi, 'theta_deg': 180.0}

def _envuelve(d, periodo=None):
    if periodo is None:
        return d
    return (d + periodo / 2.0) % periodo - periodo / 2.0

def _mad(a, eje=None, periodo=None):
    med = np.ma.median(a, axis=eje)
    exp = med if eje is None else np.expand_dims(med, eje)
    return np.ma.median(np.ma.abs(_envuelve(a - exp, periodo)), axis=eje) * 1.4826

def _pegados(m):
    """Fraccion de celdas clavadas en el extremo del propio parametro.

    Un parametro recortado a su limite fisico (§8) da EXACTAMENTE el mismo
    numero en muchas celdas. Ahi no hay nada medido, y su «dispersion» es
    cero por construccion: el veredicto de esa fila no significa nada.
    """
    v = m.compressed()
    if v.size == 0:
        return 1.0
    lo, hi = float(v.min()), float(v.max())
    tol = max(1e-12, 1e-9 * max(abs(lo), abs(hi)))
    return float(np.mean((np.abs(v - lo) <= tol) | (np.abs(v - hi) <= tol)))

def _rejilla(forma, campo):
    """Matriz (cubo, lambda) enmascarada donde el ajuste no convergio."""
    lam = LAM_MAPA[forma]
    m = np.full((len(ETQ_MAPA), len(lam)), np.nan)
    icubo = {e: i for i, e in enumerate(ETQ_MAPA)}
    ilam = {l: j for j, l in enumerate(lam)}
    for r in FILAS_MAPA:
        if r['forma'] != forma or r['status'] != 'ok':
            continue
        j = ilam.get(round(r['lambda_A'], 1))
        if j is not None and campo in r:
            m[icubo[r['cubo']], j] = r[campo]
    return np.ma.masked_invalid(m)

def _dibuja_mapa(forma, campos):
    campos = [c for c in campos if not _rejilla(forma, c).mask.all()]
    if not campos:
        print(f'{forma}: ningún ajuste convergido, no hay mapa que dibujar.')
        return
    n = len(campos); fil = (n + 1) // 2
    fig, axes = plt.subplots(fil, 2, figsize=(12, 1.9 * fil + 0.6))
    axes = np.atleast_1d(axes).ravel()
    # La frontera entre OB, para separarlos con una línea.
    _obs = [str(CABECERAS[e]['ob_id']) for e in ETQ_MAPA]
    cortes = [i for i in range(1, len(_obs)) if _obs[i] != _obs[i - 1]]
    for ax, campo in zip(axes, campos):
        m = _rejilla(forma, campo)
        v = m.compressed()
        if campo == 'C' and (v > 0).all():
            m = np.ma.masked_invalid(np.log10(m)); v = m.compressed()
            etiqueta = 'log10(C)'
        else:
            etiqueta = campo
        med = float(np.median(v))
        # Limites robustos y SIMETRICOS: el centro de la escala es la
        # mediana, asi que el gris del medio significa «como siempre».
        d = float(np.percentile(np.abs(v - med), 98)) or 1e-12
        cmap = plt.get_cmap('coolwarm').copy(); cmap.set_bad('#d9d9d9')
        im = ax.imshow(m, aspect='auto', cmap=cmap, vmin=med - d, vmax=med + d,
                       interpolation='nearest', origin='upper',
                       extent=[LAM_MAPA[forma][0], LAM_MAPA[forma][-1],
                               len(ETQ_MAPA) - 0.5, -0.5])
        for c in cortes:
            ax.axhline(c - 0.5, color='k', lw=1.0)
        ax.set_yticks(range(len(ETQ_MAPA)))
        ax.set_yticklabels(ETQ_MAPA, fontsize=5.5)
        ax.set_xlabel('λ [Å]', fontsize=8)
        ax.set_title(etiqueta, fontsize=9)
        ax.tick_params(labelsize=7)
        fig.colorbar(im, ax=ax, pad=0.01, fraction=0.04).ax.tick_params(labelsize=6)
    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle(f'{forma}: desviación respecto de la mediana '
                 '(gris claro = ajuste no convergido)', fontsize=11)
    fig.tight_layout(); plt.show()

if FILAS_MAPA:
    _dibuja_mapa('psfao', PARAMS_PSFAO)
    _dibuja_mapa('moffat', PARAMS_MOFFAT)

    # El residuo de anillo es una MAGNITUD (menos es mejor), no una
    # desviación: escala secuencial de un solo tono, no divergente.
    fig, axes = plt.subplots(1, 2, figsize=(12, 2.4 + 0.10 * len(ETQ_MAPA)))
    for ax, forma in zip(axes, ('psfao', 'moffat')):
        m = _rejilla(forma, 'ring_residual_pct')
        if m.mask.all():
            ax.set_visible(False); continue
        cmap = plt.get_cmap('Blues').copy(); cmap.set_bad('#d9d9d9')
        im = ax.imshow(m, aspect='auto', cmap=cmap, interpolation='nearest',
                       vmin=float(np.percentile(m.compressed(), 2)),
                       vmax=float(np.percentile(m.compressed(), 98)),
                       extent=[LAM_MAPA[forma][0], LAM_MAPA[forma][-1],
                               len(ETQ_MAPA) - 0.5, -0.5])
        ax.set_yticks(range(len(ETQ_MAPA)))
        ax.set_yticklabels(ETQ_MAPA, fontsize=5.5)
        ax.set_xlabel('λ [Å]', fontsize=8); ax.set_title(forma, fontsize=9)
        ax.tick_params(labelsize=7)
        fig.colorbar(im, ax=ax, pad=0.01, fraction=0.04).ax.tick_params(labelsize=6)
    fig.suptitle('residuo de anillo [%] — la comparación directa entre las dos formas',
                 fontsize=11)
    fig.tight_layout(); plt.show()


### 10.d · `theta` contra el ángulo del rotador

Un panel barato que interpreta uno de los siete. Los cubos salen todos con el norte arriba, pero el rotador del instrumento se mueve entre exposiciones: si la elongación que mide la PSF fuera **instrumental**, `theta` tendría que seguir a `ADA POSANG`; si es del cielo (o ruido de un parámetro mal determinado), no.


In [ ]:
if FILAS_MAPA and any(CABECERAS[e]['posang'] is not None for e in ETQ_MAPA):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    for ax, (forma, campo) in zip(axes, (('psfao', 'theta'), ('moffat', 'theta_deg'))):
        m = _rejilla(forma, campo)
        if m.mask.all():
            ax.set_visible(False); continue
        pa = np.array([CABECERAS[e]['posang'] if CABECERAS[e]['posang'] is not None
                       else np.nan for e in ETQ_MAPA], dtype=float)
        med = np.ma.median(m, axis=1)
        # Envuelto a su periodo: es una orientación, no un número cualquiera.
        dis = _mad(m, eje=1, periodo=PERIODO.get(campo))
        ax.errorbar(pa, med.filled(np.nan), yerr=dis.filled(np.nan),
                    ls='none', marker='o', ms=5, color=COLOR_CUBO[0],
                    capsize=2, elinewidth=0.8)
        ax.set_xlabel('ESO ADA POSANG [°]', fontsize=9)
        ax.set_ylabel(f'{campo} (mediana en λ)', fontsize=9)
        ax.set_title(forma, fontsize=9); ax.grid(alpha=0.25, lw=0.5)
        _v = med.filled(np.nan)
        _f = np.isfinite(pa) & np.isfinite(_v)
        if _f.sum() > 2:
            # Desenvuelto contra el primero: correlacionar ángulos crudos
            # cuenta un salto de periodo como una pendiente enorme.
            _ref = _v[_f][0]
            _u = _ref + _envuelve(_v[_f] - _ref, PERIODO.get(campo))
            _r = float(np.corrcoef(pa[_f], _u)[0, 1])
            ax.annotate(f'r = {_r:+.2f}', xy=(0.03, 0.92), xycoords='axes fraction',
                        fontsize=8)
    fig.suptitle('¿sigue la elongación al rotador?  (barras: MAD en λ)', fontsize=10)
    fig.tight_layout(); plt.show()
elif FILAS_MAPA:
    print('ninguna exposición declara ESO ADA POSANG: no hay nada que contrastar.')


### 10.e · La respuesta, con número

**El discriminante es la REPRODUCIBILIDAD, no la amplitud.** A cada cubo se le quita su propia mediana —para matar el desplazamiento entre exposiciones— y se pregunta si lo que queda, el perfil en λ, **es el mismo en todas**. El ruido no se repite; la física sí. Se mide con la `r` media entre los perfiles de todos los pares de cubos.

> **Por qué no vale comparar amplitudes, que es lo que esta celda hacía antes.** El primer diseño enfrentaba «cuánto salta en λ» contra «cuánto se dispersa entre exposiciones» y llamaba ruido a lo primero si era menor. La hipótesis nula estaba mal: la dispersión entre exposiciones **no es ruido de medida**, es variación real de la PSF —el seeing y la AO cambian de una exposición a otra—, así que sobreestima el suelo de ruido y hace demasiado fácil llamar «ruido» a estructura cromática de verdad. Le pasó a `r0`, que se llevó un «indistinguible del ruido» teniendo la ley de Fried entera dentro.

> **Y el «salto en λ» se medía mal.** Era el **MAD de las diferencias** entre bins contiguos, que es la *dispersión* de los saltos, no su *tamaño*: para una rampa monótona perfecta todos los saltos son iguales, su MAD es exactamente cero y la tabla la etiquetaba «sin estructura en λ». Ahora es la **mediana del valor absoluto** del salto. Las dos columnas de amplitud siguen ahí, pero como contexto, no como veredicto.

Dos correcciones más que la tabla aplica y sin las cuales engaña:

* **Las orientaciones van envueltas a su periodo** (`theta`, `theta_deg`). Una elipse a 89° y otra a −89° distan 2°, no 178°; sin envolver, `theta` salía con una dispersión del 200 % y un veredicto de «estructura real» que era el salto de periodo. Su escala de referencia es medio periodo, no la mediana.
* **La columna `pegado`** es la fracción de celdas clavadas exactamente en el extremo del parámetro. Un parámetro recortado a su límite físico (§8) repite el mismo número en muchas celdas, su dispersión entre exposiciones se va a cero y el cociente se dispara sin que haya nada medido — a `C` le sale un cociente de seis cifras con un 0.00 % de dispersión entre exposiciones. Por encima del 25 % pegado (o con la dispersión por debajo del 0.1 % de la escala) la fila dice `DEGENERADO` en vez de dar un veredicto falso.

Y un **control físico** al final, que es lo que convierte «el perfil se repite» en «y además es el perfil que toca»: `r0` es el parámetro de Fried y tiene que ir como **λ^(6/5)**. Si el ajuste lo recupera exposición por exposición, la estructura cromática de `r0` no es que sea real: es que es *conocida*, y un modelo que la imponga en vez de re-ajustarla bin a bin no pierde nada.


In [ ]:
def _perfiles_en_lambda(forma, campo):
    """Un perfil en λ por cubo, SIN el desplazamiento propio de cada uno.

    Restar la mediana de cada cubo quita justo lo que no interesa —que esa
    exposición tuviera mejor o peor seeing— y deja la forma en λ, que es lo
    que se compara entre exposiciones. Solo viajan los cubos con el perfil
    completo, para que la correlación se calcule sobre las mismas columnas.
    """
    m = _rejilla(forma, campo)
    per = PERIODO.get(campo)
    if per is not None:
        # Desenvolver cada perfil contra su primera columna: si no, un cruce
        # de periodo dentro del perfil se lee como un salto enorme.
        m = m[:, :1] + _envuelve(m - m[:, :1], per)
    p = m - np.ma.median(m, axis=1)[:, None]
    completos = np.asarray(np.ma.count(p, axis=1) == p.shape[1])
    return p, completos

def _reproducibilidad(forma, campo):
    """`r` media entre los perfiles en λ de todos los pares de cubos.

    Cerca de 1: la misma forma en λ en todas las exposiciones -> es física.
    Cerca de 0: cada exposición tiene sus propios meneos -> es ruido.
    """
    p, completos = _perfiles_en_lambda(forma, campo)
    if completos.sum() < 3 or p.shape[1] < 3:
        return float('nan')
    x = np.asarray(p[completos], dtype=float)
    if np.allclose(x.std(axis=1), 0):
        return float('nan')
    with np.errstate(invalid='ignore'):
        c = np.corrcoef(x)
    return float(np.nanmean(c[np.triu_indices_from(c, 1)]))

if FILAS_MAPA:
    for forma, campos in (('psfao', PARAMS_PSFAO), ('moffat', PARAMS_MOFFAT)):
        print(f'\n{forma}')
        print(f'{"parámetro":>12s}{"|mediana|":>12s}{"entre exp.":>12s}'
              f'{"|salto| λ":>12s}{"r perfil":>10s}{"pegado":>8s}   veredicto')
        for campo in campos:
            m = _rejilla(forma, campo)
            if m.mask.all() or m.shape[0] < 2 or m.shape[1] < 2:
                continue
            per = PERIODO.get(campo)
            esc = (per / 2.0) if per else (float(np.ma.median(np.ma.abs(m))) or 1.0)
            # Contexto 1: cuánto cambia la PSF de una exposición a otra. OJO,
            # esto NO es ruido de medida: el seeing y la AO cambian de verdad.
            entre = float(np.ma.median(_mad(m, eje=0, periodo=per)))
            # Contexto 2: el TAMAÑO típico del salto entre bins contiguos. La
            # resta va explícita porque `np.ma.diff` deja los huecos como NaN
            # sin enmascararlos. Y es la mediana del |salto|, no su MAD: el
            # MAD mide la dispersión de los saltos, y en una rampa monótona
            # —estructura en λ perfecta— todos son iguales y su MAD es CERO.
            dif = np.ma.abs(_envuelve(m[:, 1:] - m[:, :-1], per))
            enlam = float(np.ma.median(np.ma.median(dif, axis=1)))
            if not np.isfinite(entre) or not np.isfinite(enlam):
                continue
            peg = _pegados(m)
            r = _reproducibilidad(forma, campo)
            # Un parámetro clavado en su límite repite el MISMO número en
            # muchas celdas: no hay perfil que correlacionar. Le pasa a `C`.
            if peg > 0.25 or entre / esc < 1e-3:
                v = 'DEGENERADO: pegado al límite, no hay nada medido'
            elif not np.isfinite(r):
                v = 'sin cubos completos suficientes para decidir'
            elif r > 0.7:
                v = 'CROMÁTICA REAL: el perfil se repite en las exposiciones'
            elif r < 0.3:
                v = 'ruido: el perfil no se repite'
            else:
                v = 'ambiguo'
            print(f'{campo:>12s}{esc:12.4g}{100 * entre / esc:11.2f}%'
                  f'{100 * enlam / esc:11.2f}%{r:10.2f}{100 * peg:7.0f}%   {v}')
        if any(c in PERIODO for c in campos):
            print('   (las orientaciones van envueltas a su periodo y su escala es'
                  ' medio periodo,')
            print('    no la mediana: 89° y −89° distan 2°, no 178°)')
    print()
    print('`entre exp.` es contexto, no el suelo de ruido: entre exposiciones la PSF')
    print('cambia DE VERDAD (seeing, AO). Por eso el veredicto lo da `r perfil`.')

    # Control físico: r0 es el parámetro de Fried y debe ir como λ^(6/5).
    # Es lo que convierte «el perfil se repite» en «y es el que toca».
    _m = _rejilla('psfao', 'r0')
    if not _m.mask.all() and len(LAM_MAPA['psfao']) >= 5:
        _lam = np.asarray(LAM_MAPA['psfao'], dtype=float)
        _exp, _rr = [], []
        for _k in range(_m.shape[0]):
            _y = np.ma.filled(_m[_k], np.nan)
            _ok = np.isfinite(_y) & (_y > 0)
            if _ok.sum() < 5:
                continue
            _lx, _ly = np.log(_lam[_ok]), np.log(_y[_ok])
            _exp.append(float(np.polyfit(_lx, _ly, 1)[0]))
            _rr.append(float(np.corrcoef(_lx, _ly)[0, 1]))
        if len(_exp) >= 3:
            _e, _s = float(np.median(_exp)), float(np.std(_exp))
            print(f'control físico — r0 ∝ λ^p (ley de Fried, p = 6/5 = 1.200):')
            print(f'   p medido sobre {len(_exp)} exposiciones: {_e:.3f} ± {_s:.3f}'
                  f'   ({abs(_e - 1.2) / _s if _s > 0 else float("inf"):.1f} σ'
                  ' de la teoría)')
            print(f'   r de log-log, mediana: {np.median(_rr):.4f}')
            if abs(_e - 1.2) < max(3 * _s, 0.1):
                print('   -> la dependencia cromática de r0 no es que sea real:'
                      ' es CONOCIDA.')
                print('      Un modelo que la imponga en vez de reajustarla bin a'
                      ' bin no pierde nada,')
                print('      y se ahorra los escalones que la §8 mide al'
                      ' discretizarla.')

    # Y cuántos ajustes se estancan por cubo: si dominan, la salida es
    # subir BIN_A_MAPA, no seguir mirando estos números.
    print('\najustes que NO convergen, por cubo (si esto es alto, sube BIN_A_MAPA:')
    print('una exposición suelta tiene mucha menos señal que el combinado)')
    for forma in ('psfao', 'moffat'):
        _por = [sum(1 for r in FILAS_MAPA
                    if r['cubo'] == e and r['forma'] == forma and r['status'] != 'ok')
                for e in ETQ_MAPA]
        _tot = [sum(1 for r in FILAS_MAPA if r['cubo'] == e and r['forma'] == forma)
                for e in ETQ_MAPA]
        if sum(_tot):
            print(f'   {forma:7s} mediana {np.median(_por):.0f} de'
                  f' {np.median(_tot):.0f} bins · peor cubo'
                  f' {ETQ_MAPA[int(np.argmax(_por))]} con {max(_por)}')


## 11 · El híbrido que la cadena aplica y el modelo no lleva

C1 tiene un término híbrido (spec §3.5): cuando el residuo de anillo falla en más del 20 % de los bins, añade a cada modelo por bin la **mediana azimutal del residuo**. En este run se aplicó, y el número que el QC publica como `companion_ring_metric.residual_pct_median` es el de **después** de aplicarlo.

El problema es dónde acaba ese término. Los perfiles se escriben en `psf_hybrid_residual.fits`, y en `psf_model.json` lo único que queda es `"hybrid": true` — una bandera. `_evaluate_psfao` no la mira. La celda lo demuestra en vez de argumentarlo: evalúa el modelo con la bandera y sin ella, y compara.

Importa porque ese número viaja: D2 lo lee (`_psf_frac_from_qc`) y lo mete en el presupuesto de error como `sys_psf`, en cuadratura con el estadístico, y de ahí pasa a `flux_err_total` y a G2/G3. **Aquí solo se mide.**


In [ ]:
from copy import deepcopy

_sin = deepcopy(PSF_MODEL); _sin['hybrid'] = False
_dy, _dx = np.mgrid[-20:21, -20:21].astype(float)
_igual = True
for w in np.linspace(WAVE_ALL.min(), WAVE_ALL.max(), 9):
    a = evaluate_psf_model(PSF_MODEL, float(w), _dy, _dx)
    b = evaluate_psf_model(_sin, float(w), _dy, _dx)
    _igual &= bool(np.array_equal(a, b, equal_nan=True))
print('el modelo evaluado con hybrid=True y con hybrid=False es',
      'BIT A BIT IDÉNTICO' if _igual else 'distinto')
print('  -> la bandera no entra en la evaluación: el término híbrido NO viaja')
print('     con el modelo, solo a psf_hybrid_residual.fits.')
_perfil_fits = SD / 'psf_hybrid_residual.fits'
print('     ese fichero', 'existe' if _perfil_fits.exists() else 'no está',
      'y no lo lee ninguna etapa.')

_sufijo = '_canonical' if ES_PSFAO else ''
_ok_csv = [r for r in FILAS if str(r.get('status', 'ok')) == 'ok'
           and f'ring_residual_pct{_sufijo}' in r]
_antes = np.array([float(r[f'ring_residual_pct{_sufijo}']) for r in _ok_csv])
_despues = np.array([float(r[f'ring_residual_pct_after_hybrid{_sufijo}'])
                     for r in _ok_csv
                     if f'ring_residual_pct_after_hybrid{_sufijo}' in r])
print()
print('el residuo de anillo, recalculado desde el CSV por bin:')
print(f'   antes del híbrido : mediana {np.nanmedian(_antes):6.2f} %'
      f'  (el QC lo publica en model_comparison.{FORMA_ELEGIDA} ='
      f' {QC_C1["model_comparison"][FORMA_ELEGIDA]["ring_residual_pct_median"]:.2f} %)')
if _despues.size:
    print(f'   después           : mediana {np.nanmedian(_despues):6.2f} %'
          f'  (el QC lo publica en companion_ring_metric ='
          f' {QC_C1["companion_ring_metric"]["residual_pct_median"]:.2f} %)')
else:
    print('   el híbrido no se aplicó en este run: los dos números coinciden')
print()
# Lo que D2 lee de verdad es esta clave (`_psf_frac_from_qc`), no la del
# bloque `downstream_decision`, que se añadió a mano y ningún código mira.
_frac = float(QC_C1['companion_ring_metric']['residual_pct_median']) / 100.0
_declarado = QC_C1.get('downstream_decision', {}).get('psf_systematic_fraction')
print(f'lo que D2 propaga como sys_psf: {100 * _frac:.2f} %'
      ' (companion_ring_metric.residual_pct_median)')
if _declarado is not None:
    print(f'   y el QC declara psf_systematic_fraction ='
          f' {100 * float(_declarado):.2f} %, aprobado el'
          f' {QC_C1["downstream_decision"].get("approved_utc", "?")}')
    print('   (ese bloque lo escribió una persona: ningún código de musepipe lo')
    print('    pone, así que re-ejecutar C1 lo borra)')
print(f'lo que el modelo entregado consigue de verdad:'
      f' {np.nanmedian(_antes):.2f} %')
if _despues.size:
    print()
    print('los dos números describen cosas distintas y solo uno viaja aguas abajo.')
else:
    print('   (aquí coinciden: sin híbrido, lo publicado ES lo que el modelo hace)')


In [ ]:
# Y el híbrido, aplicado de verdad sobre los bins reajustados en la §6, con la
# función de la cadena: debe reproducir la columna `*_after_hybrid_canonical`.
_mids = sorted(RECONS)
if not _mids:
    print('ningún bin reajustado convergió: sube PASO_BINS o baja INCLUIR_FALLIDOS')
else:
    _imgs = [RECONS[m][0] for m in _mids]
    _mods = [RECONS[m][1] for m in _mids]
    _msk = [source_mask(_imgs[0].shape, [COMP_YX], MASK_RADIUS_PX) for _ in _mids]
    _rings = [companion_ring_metric(i, m, STAR_YX, COMP_YX,
                                    width_px=RING_WIDTH_PX,
                                    source_exclusion_radius_px=MASK_RADIUS_PX)['median_pct']
              for i, m in zip(_imgs, _mods)]
    _, _perf, _rad, _aplicado, _after, _after_p90 = _apply_hybrid(
        _rings, _imgs, _mods, _msk, STAR_YX, COMP_YX, FWHM_MED, E01)
    print(f'híbrido sobre {len(_mids)} bins: aplicado = {_aplicado}'
          f' (FWHM mediana de Moffat = {FWHM_MED:.2f} px)')
    print()
    print(f'{"λ [Å]":>9s} {"antes":>8s} {"después":>9s} {"CSV después":>12s} {"|Δ|":>10s}')
    _peor_h = 0.0
    for m, antes, desp in zip(_mids, _rings, _after):
        ref = _por_lambda[round(float(m), 3)].get(
            f'ring_residual_pct_after_hybrid{_sufijo}')
        d = abs(desp - float(ref)) if ref is not None else np.nan
        _peor_h = max(_peor_h, 0.0 if not np.isfinite(d) else d)
        _txt_ref = f'{float(ref):12.2f}' if ref is not None else f'{"no aplicado":>12s}'
        _txt_d = f'{d:10.2e}' if np.isfinite(d) else f'{"—":>10s}'
        print(f'{m:9.1f} {antes:8.2f} {desp:9.2f} {_txt_ref} {_txt_d}')
    print()
    print(f'peor discrepancia con el CSV: {_peor_h:.2e} puntos porcentuales')


## 12 · Comparación con la cadena

El producto de C1 es `psf_model.json`, y quien lo arma es una función **pura de las filas por bin** —`build_psfao_model_document` o `build_psf_model_document`, según la forma. Esas filas están en los CSV de la etapa, escritas con `repr`, así que el ida y vuelta por texto es exacto y el documento se puede reconstruir entero **sin volver a ajustar nada y sin tocar el cubo**. Eso incluye lo que de verdad importa: en psfao, el filtro de bins `ok`, el rechazo por box3 y la `param_table` que la §8 audita; en Moffat, el grado elegido por AIC para cada parámetro.

En psfao, `hybrid` se compara aparte: el constructor siempre lo deja en `False` y es la etapa quien lo sube a `True` después (§11).

Con las perillas por defecto debe salir idéntico.


In [ ]:
import warnings as _w

with _w.catch_warnings():
    _w.simplefilter('ignore')   # maoppy avisa de `alpha < 2*df` en cada bin
    if ES_PSFAO:
        MIO, META = build_psfao_model_document(
            FILAS_PSFAO, SYSTEM, NORM_RADIUS_PX, FIT_RADIUS_PX,
            system_name=SYSTEM_NAME,
            # La rejilla de evaluación es del documento, no de las filas: se le
            # devuelve la del producto para reconstruirlo entero, y se compara.
            wave_bin_A=PSF_MODEL.get('psfao_wave_bin_A'))
    else:
        MIO = build_psf_model_document(
            [float(r['wave_center_A']) for r in FILAS_MOFFAT], FILAS_MOFFAT,
            form='moffat', norm_radius_px=NORM_RADIUS_PX,
            hybrid=bool(QC_C1.get('hybrid', {}).get('applied', False)))
        META = None

def _compara(nombre, mio, suyo, rtol=1e-9):
    mio = np.asarray(mio, dtype=np.float64)
    suyo = np.asarray(suyo, dtype=np.float64)
    if mio.shape != suyo.shape:
        print(f'  {nombre:22s} FORMA distinta {mio.shape} vs {suyo.shape}')
        return False
    if mio.size == 0:
        print(f'  {nombre:22s} vacío en los dos')
        return True
    finito = np.isfinite(mio) & np.isfinite(suyo)
    if not finito.any():
        print(f'  {nombre:22s} sin valores finitos en común')
        return False
    dif = np.abs(mio - suyo)[finito]
    iguales = np.isclose(mio[finito], suyo[finito], rtol=rtol, atol=0.0)
    print(f'  {nombre:22s} idénticos {100 * iguales.mean():6.2f}% de {finito.sum():4d}'
          f' valores | máx |Δ| = {dif.max():.3e}')
    return bool(iguales.all())

ok = True
if ES_PSFAO:
    print('param_table — la tabla que interpola `_evaluate_psfao`:')
    ok &= _compara('lambda_A', MIO['param_table']['lambda_A'],
                   PSF_MODEL['param_table']['lambda_A'])
    for n in NOMBRES:
        ok &= _compara(n, MIO['param_table'][n], PSF_MODEL['param_table'][n])
    print()
    print('smoothed_poly — la parábola que el QC anuncia (y la §8 muestra inerte):')
    for n in NOMBRES:
        ok &= _compara(n, MIO['smoothed_poly'][n], PSF_MODEL['smoothed_poly'][n])
    # `psfao_wave_bin_A` entra en la lista solo si el producto la trae: los
    # documentos anteriores a que C1 la escribiera no la tienen, y ahí la
    # comparación correcta es que ninguno de los dos la declare.
    _escalares = ('n_bins_rejected', 'norm_radius_px', 'fit_radius_px') + (
        ('psfao_wave_bin_A',) if 'psfao_wave_bin_A' in PSF_MODEL else ())
else:
    print('coefficients — los polinomios que SÍ se evalúan en esta forma:')
    for n in NOMBRES:
        _a, _b = MIO['coefficients'][n], PSF_MODEL['coefficients'][n]
        _g = (int(_a['degree']) == int(_b['degree']))
        ok &= _g
        print(f'  {n:22s} grado {_a["degree"]} vs {_b["degree"]}'
              f'  {"✓" if _g else "✗"}')
        ok &= _compara(f'  {n} coef', _a['coefficients'], _b['coefficients'])
        ok &= _compara(f'  {n} wave_ref', [_a['wave_ref_A']], [_b['wave_ref_A']])
    _escalares = ('norm_radius_px',)
print()
print('escalares:')
for clave in _escalares:
    _a, _b = MIO[clave], PSF_MODEL[clave]
    _igual_k = bool(np.isclose(float(_a), float(_b), rtol=1e-12, atol=0.0))
    ok &= _igual_k
    print(f'  {clave:22s} {_a} vs {_b}  {"✓" if _igual_k else "✗"}')
for clave in (('form', 'system', 'param_names') if ES_PSFAO else ('form',)):
    _igual_k = MIO[clave] == PSF_MODEL[clave]
    ok &= bool(_igual_k)
    print(f'  {clave:22s} {MIO[clave]} vs {PSF_MODEL[clave]}'
          f'  {"✓" if _igual_k else "✗"}')
if ES_PSFAO:
    print(f'  {"hybrid":22s} {MIO["hybrid"]} vs {PSF_MODEL["hybrid"]}'
          '  (lo sube la etapa después: no es una diferencia)')
    print()
    print(f'y las filas por bin: n_ok = {META["n_ok"]}'
          f' (el QC dice {QC_C1["fit"]["n_ok_bins"]}),'
          f' rechazados = {META["n_rejected"]}'
          f' (el QC dice {QC_C1["fit"]["n_bins_rejected"]})')
    ok &= (int(META['n_ok']) == int(QC_C1['fit']['n_ok_bins']))
    ok &= (int(META['n_rejected']) == int(QC_C1['fit']['n_bins_rejected']))
else:
    _hy = (MIO['hybrid'] == PSF_MODEL['hybrid'])
    ok &= bool(_hy)
    print(f'  {"hybrid":22s} {MIO["hybrid"]} vs {PSF_MODEL["hybrid"]}'
          f'  {"✓" if _hy else "✗"}')
print()
print('IDÉNTICO: la copia reproduce la cadena.' if ok else
      'DIFIERE — si has tocado una perilla, es lo esperado; si no, mira el aviso '
      'de la §2 (modelo más viejo que su cubo) y el chequeo de deriva.')


## 13 · Qué se lleva uno de aquí

Para un run con la forma **psfao**:

1. **El `polynomial_deg2` del QC no es lo que evalúa la cadena.** Mientras exista `param_table`, `smoothed_poly` es letra muerta (§8, §12). Cualquier plan que empiece por «cambiar el grado del suavizado» está arreglando algo que no se usa.
2. **Los escalones tienen dueño y son varios**: el redondeo de λ a 50 Å, los huecos de la tabla, el recorte de los extremos y el recorte a los límites físicos de Psfao. La §8 los cuenta uno a uno, y la §9 los traduce a la curva de crecimiento, que es la forma en que llegan a C2 y C4.
3. **Los huecos no son mala suerte**: son bins que no convergieron, todos por el mismo motivo —un único vector de arranque compartido— y por eso están juntos (§7). Cerrar los huecos empieza por ahí, no por la interpolación.
4. **`C` no tiene escalones, tiene degeneración** (§8): recorre decenas de órdenes de magnitud entre bins vecinos. Suavizarlo o interpolarlo no lo arregla; es la parametrización la que no está determinada por el dato.
5. **El híbrido no viaja con el modelo** (§11). El número que se publica y el que el modelo consigue no son el mismo, y es el publicado el que entra en el presupuesto de error de D2 → G2 → G3.

Para un run con la forma **moffat** nada de eso aplica: el modelo por canal es el polinomio evaluado en la λ exacta. Lo que hay que mirar ahí es otra cosa — el grado elegido por AIC y que `outlier_bins` se publique vacío a mano, contra la §3.3 del spec.

> Nada de esto se toca desde aquí. Rehacer el modelo obliga a re-correr C1 → C2–C6 → D2 → E → F → G, y el `psf_systematic_fraction` que se publica es una decisión científica aprobada el 2026-07-29 (`stage_e01_qc.json → downstream_decision`).
